# FBref In Memoriam: Analyzing Europe's "Top 5" Leagues in 2025/26

An exploration into what we could still pull from FBref's domestic league data in the 2025/26 season.  
Built with [soccerdata](https://github.com/probberechts/soccerdata), Plotly, and Dash!

---

## Chapter 1 - Data Collection

### 1a. Imports & Dependencies

In [1]:
# System packages
import sys
from functools import reduce
from pathlib import Path
import warnings

# Dataframes and computation
import pandas as pd
import numpy as np
from scipy import stats

# Statistical tests
from scipy.stats import spearmanr
from scipy.stats import mannwhitneyu
from scipy.stats import shapiro, skew, kurtosis

# Football data package
import soccerdata as sd

# Clustering and classification
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.neighbors import KNeighborsClassifier, NearestNeighbors

# Visualization with Plotly
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Interactive apps with Dash
from dash import Dash, dcc, html, Input, Output

[03/22/26 15:32:57] INFO     No custom team name replacements found. You can configure these in       _config.py:91
                             /Users/isaiah/soccerdata/config/teamname_replacements.json.                           

                    INFO     No custom league dict found. You can configure additional leagues in    _config.py:197
                             /Users/isaiah/soccerdata/config/league_dict.json.                                     

### 1b. Global Variables (our "constants")

In [2]:
# -- We'll create a single definition point for all constants, which will often be used on mulitple occasions --

font_family    = "Georgia, 'Times New Roman', serif" # No particular reason other than I liked this the most from the choices I was recomended 

# Position mapping by number (used for colorscale encoding)
position_map   = {'Defender': 0, 'Wingback': 1, 'Midfielder': 2, 'Forward': 3}

# Color key (used for different types of visualizations with positional distinctions)
position_colors = {
    'Forward':    '#1f77b4',
    'Midfielder': '#2ca02c',
    'Wingback':   "#ffa70e",
    'Defender':   '#d62728',
}

# Marker style for scatter plots further down the line
MARKER_STYLE = dict(
    colorscale=[[0, '#d62728'], [0.33, '#ffa70e'], [0.67, '#2ca02c'], [1, '#1f77b4']],
    showscale=True,
    colorbar=dict(title="Position", tickvals=[0, 1, 2, 3],
                  ticktext=['Defender', 'Wingback', 'Midfielder', 'Forward']),
    cmin=0, cmax=3,
    line=dict(width=0.5, color='white'),
    size=9, opacity=0.8,
)

# Position group order and column mapping
position_order = ["Defender", "Wingback", "Midfielder", "Forward"]
cog_col_map    = {
    "Defender":   "cog_defender",
    "Wingback":   "cog_wingback",
    "Midfielder": "cog_midfielder",
    "Forward":    "cog_forward",
}

# Rotation metric - confirmed from Shapiro-Wilk test/EDA in chapter 3a (minutes aren't normally distributed, thus M.A.D. preferred over Standard Deviation)
rotation_metric = "minutes_mad"
rotation_label  = "Minutes M.A.D." 

### 1c. Pulling Data from FBref

In [3]:
# Define what we want to pull in these variables: 5 leagues, current season, 8 stat categories (the most we can pull from FBref)
seasons = ["2025-2026"]
leagues = [
    "ENG-Premier League",
    "ESP-La Liga",
    "GER-Bundesliga",
    "ITA-Serie A",
    "FRA-Ligue 1",
]
stat_types = [
    "standard",
    "shooting",
    "passing",
    "passing_types",
    "goal_shot_creation",
    "defense",
    "possession",
    "misc",
]

# Create our FBref object to pull from
fb = sd.FBref(leagues=leagues, seasons=seasons)

# Load each stat category one at a time via a "for" loop
dfs = []
for st in stat_types:
    print(f"Loading {st} ...", flush=True)
    df = fb.read_player_season_stats(stat_type=st)
    print(f"  {st} loaded: {len(df)} rows", flush=True)
    dfs.append(df)

[03/22/26 15:33:01] INFO     Saving cached data to /Users/isaiah/soccerdata/data/FBref               _common.py:263

                    WARNING  /Users/isaiah/Documents/Isaiah's Python                                ]8;id=824395;file:///Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/warnings.py\warnings.py]8;;\:]8;id=75782;file:///Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/warnings.py#110\110]8;;\
                             Projects/.soccerenv/lib/python3.11/site-packages/soccerdata/fbref.py:9                
                             8: UserWarning: You are trying to scrape data for all of the Big 5                    
                             European leagues. This can be done more efficiently by setting                        
                             leagues='Big 5 European Leagues Combined'.                                            
                               warnings.warn(                                                                      
                                                                                                                   

Loading standard ...


                    WARNING  /Users/isaiah/Documents/Isaiah's Python                                ]8;id=754567;file:///Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/warnings.py\warnings.py]8;;\:]8;id=493079;file:///Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/warnings.py#110\110]8;;\
                             Projects/.soccerenv/lib/python3.11/site-packages/soccerdata/fbref.py:1                
                             65: FutureWarning: The behavior of DataFrame concatenation with empty                 
                             or all-NA entries is deprecated. In a future version, this will no                    
                             longer exclude empty or all-NA columns when determining the result                    
                             dtypes. To retain the old behavior, exclude the relevant entries                      
                             before the concat operation.                                                          
                               pd.concat(dfs)                                                                      
                                                                                                                   

  standard loaded: 2346 rows
Loading shooting ...


...

  defense loaded: 2346 rows
Loading possession ...


[03/22/26 15:33:05] WARNING  /Users/isaiah/Documents/Isaiah's Python                                ]8;id=870029;file:///Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/warnings.py\warnings.py]8;;\:]8;id=53748;file:///Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/warnings.py#110\110]8;;\
                             Projects/.soccerenv/lib/python3.11/site-packages/soccerdata/fbref.py:1                
                             65: FutureWarning: The behavior of DataFrame concatenation with empty                 
                             or all-NA entries is deprecated. In a future version, this will no                    
                             longer exclude empty or all-NA columns when determining the result                    
                             dtypes. To retain the old behavior, exclude the relevant entries                      
                             before the concat operation.                                                          
                               pd.concat(dfs)                                                                      
                                                                                                                   

  possession loaded: 2346 rows
Loading misc ...


                    WARNING  /Users/isaiah/Documents/Isaiah's Python                                ]8;id=919995;file:///Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/warnings.py\warnings.py]8;;\:]8;id=614114;file:///Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/warnings.py#110\110]8;;\
                             Projects/.soccerenv/lib/python3.11/site-packages/soccerdata/fbref.py:1                
                             65: FutureWarning: The behavior of DataFrame concatenation with empty                 
                             or all-NA entries is deprecated. In a future version, this will no                    
                             longer exclude empty or all-NA columns when determining the result                    
                             dtypes. To retain the old behavior, exclude the relevant entries                      
                             before the concat operation.                                                          
                               pd.concat(dfs)                                                                      
                                                                                                                   

  misc loaded: 2346 rows


In [4]:
# Assign each stat category to its own named variable for when we merge
standard           = dfs[0]
shooting           = dfs[1]
passing            = dfs[2]
passing_types      = dfs[3]
goal_shot_creation = dfs[4]
defense            = dfs[5]
possession         = dfs[6]
misc               = dfs[7]

---

## Chapter 2 - Player Data: Munging & Visualizations

### 2a. Merging & Cleaning

In [5]:
# Since the raw FBref pull comes with multi-index columns, we'll want to flatten them all to a single level
grouped_dfs = {
    "standard": standard,
    "shooting": shooting,
    "passing": passing,
    "passing_types": passing_types,
    "goal_shot_creation": goal_shot_creation,
    "defense": defense,
    "possession": possession,
    "misc": misc,
}

def flatten_columns(df):
    df2 = df.copy()
    df2.columns = [
        "_".join(str(c) for c in col if c)
        if isinstance(col, tuple) else str(col)
        for col in df2.columns
    ]
    return df2

for name, df in grouped_dfs.items():
    globals()[name] = flatten_columns(df)

standard.sample(3)

,,,,nation,pos,age,born,Playing Time_MP,Playing Time_Starts,Playing Time_Min,Playing Time_90s,Performance_Gls,Performance_Ast,...,Per 90 Minutes_Gls,Per 90 Minutes_Ast,Per 90 Minutes_G+A,Per 90 Minutes_G-PK,Per 90 Minutes_G+A-PK,Per 90 Minutes_xG,Per 90 Minutes_xAG,Per 90 Minutes_xG+xAG,Per 90 Minutes_npxG,Per 90 Minutes_npxG+xAG
league,season,team,player,,,,,,,,,,,,,,,,,,,,,
FRA-Ligue 1,2526,Brest,Pathé Mboup,SEN,FW,22-113,2003,13,8,675,7.5,1,0,...,0.13,0.0,0.13,0.13,0.13,0.28,0.09,0.38,0.28,0.38
ESP-La Liga,2526,Levante,Alan Matturro,URU,DF,21-067,2004,4,2,161,1.8,0,0,...,0.0,0.0,0.0,0.0,0.0,0.01,0.0,0.01,0.01,0.01
ENG-Premier League,2526,Manchester City,Gianluigi Donnarumma,ITA,GK,26-295,1999,13,13,1170,13.0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [6]:
# We'll need to remove duplicate columns across dataframes, keep only first occurrences, and merge on key columns
df_list = [
    ("standard",           standard),
    ("shooting",           shooting),
    ("passing",            passing),
    ("passing_types",      passing_types),
    ("goal_shot_creation", goal_shot_creation),
    ("defense",            defense),
    ("possession",         possession),
    ("misc",               misc),
]

key_cols = ["league", "season", "team", "player"] # This is deliberate! A player who transfers mid-season gets separate rows per team

seen_nonkey = set()
cleaned = []

for name, df in df_list:
    df2 = df.copy()
    to_drop = []
    for col in df2.columns:
        if col in key_cols:
            continue
        if col in seen_nonkey:
            to_drop.append(col)
        else:
            seen_nonkey.add(col)
    df2 = df2.drop(columns=to_drop, errors="ignore")
    print(f"{name}: dropped {len(to_drop)} duplicate cols -> shape {df2.shape}")
    cleaned.append(df2)

players_all = reduce(
    lambda left, right: left.merge(right, on=key_cols, how="inner"),
    cleaned,
)

print("Final merged shape:", players_all.shape)
players_all.sample(3)

standard: dropped 0 duplicate cols -> shape (2346, 33)
shooting: dropped 6 duplicate cols -> shape (2346, 16)
passing: dropped 5 duplicate cols -> shape (2346, 23)
passing_types: dropped 5 duplicate cols -> shape (2346, 15)
goal_shot_creation: dropped 5 duplicate cols -> shape (2346, 16)
defense: dropped 5 duplicate cols -> shape (2346, 16)
possession: dropped 5 duplicate cols -> shape (2346, 22)
misc: dropped 7 duplicate cols -> shape (2346, 14)
Final merged shape: (2346, 155)


,,,,nation,pos,age,born,Playing Time_MP,Playing Time_Starts,Playing Time_Min,Playing Time_90s,Performance_Gls,Performance_Ast,...,Performance_Crs,Performance_Int,Performance_TklW,Performance_PKwon,Performance_PKcon,Performance_OG,Performance_Recov,Aerial Duels_Won,Aerial Duels_Lost,Aerial Duels_Won%
league,season,team,player,,,,,,,,,,,,,,,,,,,,,
FRA-Ligue 1,2526,Lyon,Afonso Moreira,POR,FW,20-273,2005,14,5,565,6.3,2,4,...,52,4,9,0,0,0,32,1,9,10.0
ENG-Premier League,2526,Brighton,Diego Gómez,PAR,"MF,FW",22-265,2003,15,11,959,10.7,2,0,...,9,7,22,0,0,0,41,18,14,56.3
FRA-Ligue 1,2526,Marseille,Nayef Aguerd,MAR,DF,29-262,1996,11,11,963,10.7,1,0,...,1,13,11,0,0,0,53,28,16,63.6


In [7]:
# Precautionary measure: if key columns are still in the index and not the columns, let's promote them to align

if set(['league', 'season', 'team', 'player']).issubset(set(players_all.index.names)):
    players_all = players_all.reset_index(drop=False)

# Renaming columns for interpretability!
rename_map = {
    'nation':'Nationality', 'pos':'Position', 'age':'Age', 'born':'Birth Year',
    'Playing Time_MP':'Matches Played', 'Playing Time_Starts':'Matches Started',
    'Playing Time_Min':'Minutes Played', 'Playing Time_90s':'90s Played',
    'Performance_Gls':'Goals', 'Performance_Ast':'Assists',
    'Performance_G+A':'Goals + Assists', 'Performance_G-PK':'Non-Penalty Goals',
    'Performance_PK':'Penalties Scored', 'Performance_PKatt':'Penalties Attempted',
    'Performance_CrdY':'Yellow Cards', 'Performance_CrdR':'Red Cards',
    'Expected_xG':'Expected Goals', 'Expected_npxG':'Non-Penalty Expected Goals',
    'Expected_xAG':'Expected Assisted Goals',
    'Expected_npxG+xAG':'Non-Penalty Expected Goals + Expected Assisted Goals',
    'Progression_PrgC':'Progressive Carries', 'Progression_PrgP':'Progressive Passes',
    'Progression_PrgR':'Progressive Passes Received',
    'Standard_Gls':'Goals', 'Standard_Sh':'Shots', 'Standard_SoT':'Shots on Target',
    'Standard_SoT%':'Shots on Target %', 'Standard_Sh/90':'Shots p90',
    'Standard_SoT/90':'Shots on Target p90', 'Standard_G/Sh':'Goals per Shot',
    'Standard_G/SoT':'Goals per Shot on Target', 'Standard_Dist':'Average Shot Distance',
    'Standard_FK':'Free Kicks Scored', 'Standard_PK':'Penalties Scored',
    'Standard_PKatt':'Penalties Attempted',
    'Expected_npxG/Sh':'Expected Non-Penalty Goals per Shot',
    'Expected_G-xG':'xG Overperformance', 'Expected_np:G-xG':'Non-Penalty xG Overperformance',
    'Total_Cmp':'Total Passes Completed', 'Total_Att':'Total Passes Attempted',
    'Total_Cmp%':'Total Pass Completion %', 'Total_TotDist':'Total Passing Distance',
    'Total_PrgDist':'Progressive Passing Distance',
    'Short_Cmp':'Short Passes Completed', 'Short_Att':'Short Passes Attempted',
    'Short_Cmp%':'Short Pass Completion %', 'Medium_Cmp':'Medium Passes Completed',
    'Medium_Att':'Medium Passes Attempted', 'Medium_Cmp%':'Medium Pass Completion %',
    'Long_Cmp':'Long Passes Completed', 'Long_Att':'Long Passes Attempted',
    'Long_Cmp%':'Long Pass Completion %', 'Ast':'Assists', 'xAG':'Expected Assisted Goals',
    'Expected_xA':'Expected Assists', 'Expected_A-xAG':'xAG Overperformance',
    'KP':'Key Passes', '1/3':'Passes into Final Third',
    'PPA':'Passes into Penalty Area', 'CrsPA':'Crosses into Penalty Area',
    'PrgP':'Progressive Passes', 'Att':'Total Passes Attempted',
    'Pass Types_Live':'Live Ball Passes', 'Pass Types_Dead':'Dead Ball Passes',
    'Pass Types_FK':'Free Kick Passes', 'Pass Types_TB':'Through Balls',
    'Pass Types_Sw':'Switches', 'Pass Types_Crs':'Crosses',
    'Pass Types_TI':'Throw-Ins', 'Pass Types_CK':'Corner Kicks',
    'Corner Kicks_In':'Inswinging Corner Kicks', 'Corner Kicks_Out':'Outswinging Corner Kicks',
    'Corner Kicks_Str':'Straight Corner Kicks',
    'Outcomes_Cmp':'Passes Completed', 'Outcomes_Off':'Passes Offsides',
    'Outcomes_Blocks':'Passes Blocked',
    'SCA_SCA':'Shot-Creating Actions', 'SCA_SCA90':'Shot-Creating Actions p90',
    'SCA Types_PassLive':'Live Ball Pass SCAs', 'SCA Types_PassDead':'Dead Ball Pass SCAs',
    'SCA Types_TO':'Take-On SCAs', 'SCA Types_Sh':'Shot SCAs',
    'SCA Types_Fld':'Drawn Foul SCAs', 'SCA Types_Def':'Defensive SCAs',
    'GCA_GCA':'Goal-Creating Actions', 'GCA_GCA90':'Goal-Creating Actions p90',
    'GCA Types_PassLive':'Live Ball Pass GCAs', 'GCA Types_PassDead':'Dead Ball Pass GCAs',
    'GCA Types_TO':'Take-On GCAs', 'GCA Types_Sh':'Shot GCAs',
    'GCA Types_Fld':'Drawn Foul GCAs', 'GCA Types_Def':'Defensive GCAs',
    'Tackles_Tkl':'Tackles', 'Tackles_TklW':'Possession-Winning Tackles',
    'Tackles_Def 3rd':'Defensive 3rd Tackles', 'Tackles_Mid 3rd':'Midfield 3rd Tackles',
    'Tackles_Att 3rd':'Attacking 3rd Tackles',
    'Challenges_Tkl':'Dribbles Tackled', 'Challenges_Att':'Attempted Dribbles Tackled',
    'Challenges_Tkl%':'Dribblers Tackled %', 'Challenges_Lost':'Challenges Lost',
    'Blocks_Blocks':'Blocks', 'Blocks_Sh':'Blocked Shots', 'Blocks_Pass':'Blocked Passes',
    'Int':'Interceptions', 'Tkl+Int':'Tackles + Interceptions',
    'Clr':'Clearances', 'Err':'Errors',
    'Touches_Touches':'Touches', 'Touches_Def Pen':'Touches in Defensive Penalty Box',
    'Touches_Def 3rd':'Touches in Defensive 3rd', 'Touches_Mid 3rd':'Touches in Midfield 3rd',
    'Touches_Att 3rd':'Touches in Attacking 3rd',
    'Touches_Att Pen':'Touches in Attacking Penalty Box',
    'Touches_Live':'Live Ball Touches',
    'Take-Ons_Att':'Take-Ons Attempted', 'Take-Ons_Succ':'Successful Take-Ons',
    'Take-Ons_Succ%':'Take-On Success %', 'Take-Ons_Tkld':'Times Tackled During Take-On',
    'Take-Ons_Tkld%':'Tackled During Take-On %',
    'Carries_Carries':'Carries', 'Carries_TotDist':'Total Carrying Distance',
    'Carries_PrgDist':'Progressive Carrying Distance', 'Carries_PrgC':'Progressive Carries',
    'Carries_1/3':'Carries into Final Third', 'Carries_CPA':'Carries into Penalty Area',
    'Carries_Mis':'Miscontrols', 'Carries_Dis':'Dispossessed',
    'Receiving_Rec':'Passes Received', 'Receiving_PrgR':'Progressive Passes Received',
    'Performance_2CrdY':'Second Yellow Cards', 'Performance_Fls':'Fouls Committed',
    'Performance_Fld':'Fouls Won', 'Performance_Off':'Offsides',
    'Performance_Crs':'Crosses', 'Performance_Int':'Interceptions',
    'Performance_TklW':'Tackles Won', 'Performance_PKwon':'Penalties Won',
    'Performance_PKcon':'Penalties Conceded', 'Performance_OG':'Own Goals',
    'Performance_Recov':'Ball Recoveries',
    'Aerial Duels_Won':'Aerial Duels Won', 'Aerial Duels_Lost':'Aerial Duels Lost',
    'Aerial Duels_Won%':'Aerial Win %',
}

players_all = players_all.rename(columns=rename_map)
players_all = players_all.loc[:, ~players_all.columns.duplicated(keep='first')]

# I'll also drop any pre-existing "per 90 minutes" columns , since we'll do sweeping creation of our own anyways
p90_cols_to_drop = [col for col in players_all.columns if 'p90' in col]
players_all = players_all.drop(columns=p90_cols_to_drop)

# Assign a clean numeric Player ID index
players_all['ID'] = range(1, len(players_all) + 1)
players_all = players_all.set_index('ID')

# Now, we'll generate "per 90" versions of all eligible features
exclude_columns = [
    'league', 'season', 'team', 'player', 'Nationality', 'Position',
    'Age', 'Birth Year', 'Matches Played', 'Matches Started',
    'Minutes Played', '90s Played'
]
exclude_patterns = ['_total', 'COG', 'per 100', '%', 'p90', 'Overperformance', 'Average']

per90_feature_list = []
for col in players_all.columns:
    if col in exclude_columns:
        continue
    if any(pattern in col for pattern in exclude_patterns):
        continue
    if not pd.api.types.is_numeric_dtype(players_all[col]):
        continue
    per90_feature_list.append(col)

for col in per90_feature_list:
    players_all[col + " per 90"] = players_all[col] / players_all["90s Played"]

players_all.sample(3)

,league,season,team,player,Nationality,Position,Age,Birth Year,Matches Played,Matches Started,...,Fouls Committed per 90,Fouls Won per 90,Offsides per 90,Tackles Won per 90,Penalties Won per 90,Penalties Conceded per 90,Own Goals per 90,Ball Recoveries per 90,Aerial Duels Won per 90,Aerial Duels Lost per 90
ID,,,,,,,,,,,,,,,,,,,,,
1663,GER-Bundesliga,2526,Leverkusen,Aleix García,ESP,MF,28-172,1997,14,13,...,0.15748,0.472441,0.07874,0.314961,0.0,0.0,0.0,4.409449,0.23622,0.23622
333,ENG-Premier League,2526,Newcastle Utd,Kieran Trippier,ENG,DF,35-089,1990,9,8,...,1.666667,1.025641,0.0,0.769231,0.0,0.0,0.0,4.487179,1.153846,0.897436
1482,GER-Bundesliga,2526,Dortmund,Jobe Bellingham,ENG,MF,20-085,2005,14,5,...,1.111111,1.481481,0.0,1.666667,0.0,0.0,0.0,5.37037,1.296296,0.925926


In [8]:
# Let's also make some calculated fields that will be insightful for future analysis!
players_all = players_all.copy()

players_all["Possession-Winning Tackle %"] = (
    players_all["Possession-Winning Tackles"] / players_all["Tackles"]
)
players_all["Non-Penalty Goals per Shot"] = (
    players_all["Non-Penalty Goals"] / players_all["Shots"]
)
players_all["Total Progression Distance"] = (
    players_all["Progressive Passing Distance"] + players_all["Progressive Carrying Distance"]
)
players_all["Miscontrols per 100 Touches"] = (
    players_all["Miscontrols"] / players_all["Touches"] * 100
)
players_all["Turnovers per 100 Touches"] = (
    (players_all["Miscontrols"] + players_all["Dispossessed"] + players_all["Errors"])
    / players_all["Touches"] * 100
)
players_all["Switches per 100 Touches"] = (
    players_all["Switches"] / players_all["Touches"] * 100
)
players_all["Progressive Passes per 100 Touches"] = (
    players_all["Progressive Passes"] / players_all["Touches"] * 100
)
players_all["Progressive Carries per 100 Touches"] = (
    players_all["Progressive Carries"] / players_all["Touches"] * 100
)
players_all["Progressive Passing Distance per 100 Touches"] = (
    players_all["Progressive Passing Distance"] / players_all["Touches"] * 100
)
players_all["Progressive Carrying Distance per 100 Touches"] = (
    players_all["Progressive Carrying Distance"] / players_all["Touches"] * 100
)
players_all["Passes into Penalty Area per 100 Touches"] = (
    players_all["Passes into Penalty Area"] / players_all["Touches"] * 100
)
players_all["Progressive Receptions per 100 Touches"] = (
    players_all["Progressive Passes Received"] / players_all["Touches"] * 100
)
players_all["Blocks + Clearances"] = (
    players_all["Blocks"] + players_all["Clearances"]
)

# Let's sense check a few of our new calculated fields
players_all[["player", "team", "Turnovers per 100 Touches",
             "Progressive Passes per 100 Touches",
             "Progressive Carries per 100 Touches"]].sample(3)

,player,team,Turnovers per 100 Touches,Progressive Passes per 100 Touches,Progressive Carries per 100 Touches
ID,,,,,
715,Yangel Herrera,Girona,4.918033,3.278689,3.278689
1938,Jean Butez,Como,0.310078,0.310078,0.0
167,Adam Wharton,Crystal Palace,3.46908,10.859729,2.111614


### 2a(ii). Dataset Overview

Before we dive in - a quick look at what we're working with. This is the full merged dataset, all 5 leagues, all positions including GKs.

In [9]:
# Since this cell's output will contain our EDA visualizations, we'll ignore warnings for a cleaner look
warnings.filterwarnings("ignore")

print("=" * 35)
print("  Dataset Overview: players_all")
print("=" * 35)
print(f"  Rows (player-seasons):  {len(players_all):,}")
print(f"  Columns:                {players_all.shape[1]:,}")
print(f"  Leagues:                {players_all['league'].nunique()}")
print(f"  Seasons:                {players_all['season'].nunique()}")
print(f"  Teams:                  {players_all['team'].nunique()}")
print(f"  Unique players:         {players_all['player'].nunique():,}")
print()

# -- League breakdown --
print("-- Players per League --")
league_counts = players_all.groupby("league")["player"].count().sort_values(ascending=False)
for league, count in league_counts.items():
    teams_in_league = players_all[players_all["league"] == league]["team"].nunique()
    print(f"  {league:<28} {count:>4} players  |  {teams_in_league} teams")
print()

# -- Position breakdown --
print("-- Raw FBref Positions --")
pos_counts = players_all["Position"].value_counts()
for pos, count in pos_counts.items():
    pct = round(count / len(players_all) * 100, 1)
    print(f"  {str(pos):<12} {count:>4}  ({pct}%)")
print()

# -- Minutes distribution --
mins = players_all["Minutes Played"].dropna()
print("-- Minutes Played Distribution --")
print(f"  Min:     {mins.min():.0f}")
print(f"  Median:  {mins.median():.0f}")
print(f"  Mean:    {mins.mean():.0f}")
print(f"  Max:     {mins.max():.0f}")
print(f"  STD:     {mins.std():.0f}")
print()
print(f"  Players with 0 minutes:      {(mins == 0).sum()}")
print(f"  Players with < 90 minutes:   {(mins < 90).sum()}")
print(f"  Players with 270+ minutes:   {(mins >= 270).sum()}  "
      f"({round((mins >= 270).sum() / len(mins) * 100, 1)}%)  ← analysis threshold")
print()

# -- Summary of nulls --
print("-- Null Value Summary --")
null_counts = players_all.isnull().sum()
null_pct    = (null_counts / len(players_all) * 100).round(1)
null_df     = pd.DataFrame({"nulls": null_counts, "pct": null_pct})
null_df     = null_df[null_df["nulls"] > 0].sort_values("pct", ascending=False)

if null_df.empty:
    print("  No null values found.")
else:
    print(f"  {len(null_df)} columns contain nulls:\\n")
    high   = null_df[null_df["pct"] >= 20]
    medium = null_df[(null_df["pct"] >= 5) & (null_df["pct"] < 20)]
    low    = null_df[null_df["pct"] < 5]
    for label, subset in [("High (≥20%)", high),
                           ("Medium (5–20%)", medium),
                           ("Low (<5%)", low)]:
        if not subset.empty:
            print(f"  {label}:")
            for col, row in subset.iterrows():
                print(f"    {col:<45} {row['nulls']:>4} nulls  ({row['pct']}%)")
            print()

# -- Visualizations --
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        "Players per League",
        "Minutes Played Distribution",
        "Age Distribution",
        "Nationality (Top 10)",
    ],
    vertical_spacing=0.18,
    horizontal_spacing=0.12,
)

# 1. Players per league bar
fig.add_trace(go.Bar(
    x=league_counts.values,
    y=league_counts.index,
    orientation="h",
    marker=dict(
        color=px.colors.qualitative.Plotly[:len(league_counts)],
        line=dict(width=0.5, color="rgba(255,255,255,0.2)"),
    ),
    hovertemplate="<b>%{y}</b><br>Players: %{x}<br><extra></extra>",
    showlegend=False,
), row=1, col=1)

# 2. Minutes played histogram
fig.add_trace(go.Histogram(
    x=players_all["Minutes Played"].dropna(),
    nbinsx=30,
    marker=dict(
        color="#4a90d9", opacity=0.8,
        line=dict(width=0.5, color="rgba(255,255,255,0.2)"),
    ),
    hovertemplate="Minutes: %{x}<br>Count: %{y}<br><extra></extra>",
    showlegend=False,
), row=1, col=2)

# 270 min threshold line - since I want to see how many players would be cutoff by this minimum
fig.add_vline(
    x=270,
    line=dict(color="rgba(245,197,24,0.7)", width=1.5, dash="dash"),
    annotation_text="270 min threshold",
    annotation_position="top right",
    annotation_font=dict(family=font_family, size=9,
                         color="rgba(245,197,24,0.8)"),
    row=1, col=2,
)

# 3. Age distribution
age_clean = players_all["Age"].dropna().astype(str).str.split("-").str[0]
age_clean = pd.to_numeric(age_clean, errors="coerce").dropna()
fig.add_trace(go.Histogram(
    x=age_clean,
    nbinsx=25,
    marker=dict(
        color="#2ca02c", opacity=0.8,
        line=dict(width=0.5, color="rgba(255,255,255,0.2)"),
    ),
    hovertemplate="Age: %{x}<br>Count: %{y}<br><extra></extra>",
    showlegend=False,
), row=2, col=1)

fig.add_vline(
    x=age_clean.mean(),
    line=dict(color="rgba(245,197,24,0.7)", width=1.5, dash="dash"),
    annotation_text=f"Mean: {age_clean.mean():.1f}",
    annotation_position="top right",
    annotation_font=dict(family=font_family, size=9,
                         color="rgba(245,197,24,0.8)"),
    row=2, col=1,
)

# 4. Top 10 nationalities
top_nations = (
    players_all["Nationality"].dropna()
    .value_counts()
    .head(10)
    .sort_values(ascending=True)
)
fig.add_trace(go.Bar(
    x=top_nations.values,
    y=top_nations.index,
    orientation="h",
    marker=dict(
        color="#ff7f0e", opacity=0.85,
        line=dict(width=0.5, color="rgba(255,255,255,0.2)"),
    ),
    hovertemplate="<b>%{y}</b><br>Players: %{x}<br><extra></extra>",
    showlegend=False,
), row=2, col=2)

fig.update_layout(
    title=dict(
        text="Exploratory Data Analysis - players_all (2025/26 Season)",
        font=dict(family=font_family, size=15, color="#f5f7fa"),
        x=0, xanchor="left",
    ),
    font=dict(family=font_family, color="#f5f7fa"),
    paper_bgcolor="#001f3f", plot_bgcolor="#001f3f",
    height=700,
    margin=dict(l=60, r=40, t=80, b=40),
)

for ax in fig.layout:
    if ax.startswith("xaxis") or ax.startswith("yaxis"):
        fig.layout[ax].update(
            showgrid=True, gridcolor="#1a2a3a", zeroline=False,
            tickfont=dict(family=font_family, color="#cdd9e5", size=10),
        )

for annotation in fig.layout.annotations:
    annotation.font.update(family=font_family, color="#cdd9e5", size=12)

fig.show()

  Dataset Overview: players_all
  Rows (player-seasons):  2,346
  Columns:                276
  Leagues:                5
  Seasons:                1
  Teams:                  96
  Unique players:         2,299

-- Players per League --
  ESP-La Liga                   500 players  |  20 teams
  ITA-Serie A                   493 players  |  20 teams
  ENG-Premier League            468 players  |  20 teams
  FRA-Ligue 1                   458 players  |  18 teams
  GER-Bundesliga                427 players  |  18 teams

-- Raw FBref Positions --
  DF            724  (30.9%)
  MF            510  (21.7%)
  FW            348  (14.8%)
  FW,MF         220  (9.4%)
  MF,FW         184  (7.8%)
  GK            145  (6.2%)
  DF,MF          95  (4.0%)
  MF,DF          65  (2.8%)
  DF,FW          37  (1.6%)
  FW,DF          18  (0.8%)

-- Minutes Played Distribution --
  Min:     1
  Median:  596
  Mean:    624
  Max:     1530
  STD:     433

  Players with 0 minutes:      0
  Players with < 90 minut

### 2b. Center of Gravity - Introduction to Concept

In [10]:
# Touch Center of Gravity (COG): weighted average of where a player's touches happen on the pitch
# Zones are scored -1 (own box) to +1 (opponent box), with midfield at 0

players_all["touches_total"] = (
    players_all["Touches in Defensive Penalty Box"] +
    players_all["Touches in Defensive 3rd"] +
    players_all["Touches in Midfield 3rd"] +
    players_all["Touches in Attacking 3rd"] +
    players_all["Touches in Attacking Penalty Box"]
)

players_all["Touch COG"] = (
    players_all["Touches in Defensive Penalty Box"] * -1.0 +
    players_all["Touches in Defensive 3rd"]         * -0.6 +
    players_all["Touches in Midfield 3rd"]           *  0.0 +
    players_all["Touches in Attacking 3rd"]          *  0.6 +
    players_all["Touches in Attacking Penalty Box"]  *  1.0
) / players_all["touches_total"]

# Tackle Center of Gravity (COG): same idea, but for where tackles are won
# Flag: only 3 tackle zones available from FBref (no penalty box split, unlike with touches)

players_all["tackles_total"] = (
    players_all["Defensive 3rd Tackles"] +
    players_all["Midfield 3rd Tackles"] +
    players_all["Attacking 3rd Tackles"]
).astype(float)

players_all["Tackle COG"] = (
    players_all["Defensive 3rd Tackles"] * -1.0 +
    players_all["Midfield 3rd Tackles"]   *  0.0 +
    players_all["Attacking 3rd Tackles"]  *  1.0
).astype(float) / players_all["tackles_total"]

players_all[["player", "team", "Position", "Touch COG", "Tackle COG"]].sample(3)

,player,team,Position,Touch COG,Tackle COG
ID,,,,,
505,Mikel Jauregizar,Athletic Club,MF,0.041517,-0.114286
2060,Federico Dimarco,Inter,DF,0.204061,-0.200000
263,Florian Wirtz,Liverpool,"MF,FW",0.26978,-0.125000


In [12]:
# Baseline player filter: outfield players only, minimum of 270 minutes (equivalent of 3 full matches)
# NaNs imputed to 0 (see note in 2a for why this is acceptable for count stats)
players_filtered = (
    players_all
    .fillna(0)
    .loc[
        (players_all["Minutes Played"] >= 270) &
        (players_all["Position"] != "GK")
    ]
    .copy()
)
print(f"players_filtered: {players_filtered.shape[0]} players")

# Three dataframes in play from here on out
# players_all      -- whole merged dataset, all positions including GKs
# players_filtered -- outfield players, 270+ minutes, used for all analyses
# scores_df        -- made later in 2g, a subset of players_filtered with scores

players_filtered: 1564 players


In [13]:
# Initial look: Touch COG vs Tackle COG scatter, colored by team, with team filter dropdown
df = players_filtered.copy()

teams_unique = sorted(df["team"].unique())
palette = px.colors.qualitative.Plotly
team_to_color = {team: palette[i % len(palette)] for i, team in enumerate(teams_unique)}
df["team_color"] = df["team"].map(team_to_color)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df["Touch COG"], y=df["Tackle COG"],
    mode="markers",
    marker=dict(size=9, opacity=0.8, color=df["team_color"]),
    text=df["player"],
    customdata=np.stack([df["team"]], axis=-1),
    hovertemplate=(
        "<b>%{text}</b><br>Team: %{customdata[0]}<br>"
        "Touch COG: %{x:.2f}<br>Tackle COG: %{y:.2f}<br><extra></extra>"
    ),
))

teams = ["All"] + teams_unique
buttons = []
for t in teams:
    mask = np.ones(len(df), dtype=bool) if t == "All" else (df["team"] == t)
    buttons.append(dict(
        label=t, method="update",
        args=[{"x": [df.loc[mask, "Touch COG"]], "y": [df.loc[mask, "Tackle COG"]],
               "text": [df.loc[mask, "player"]],
               "customdata": [np.stack([df.loc[mask, "team"]], axis=-1)],
               "marker": [dict(size=9, opacity=0.8, color=df.loc[mask, "team_color"])]}],
    ))

axis_cfg = dict(range=[-1, 1], tickvals=[-1, -0.5, 0, 0.5, 1],
                zeroline=True, zerolinewidth=1, zerolinecolor="gray")
fig.update_layout(
    updatemenus=[dict(buttons=buttons, direction="down", x=0, y=1.15,
                      showactive=True, pad={"r": 10})],
    title='Positional "Center of Gravity" Map',
    xaxis=dict(title="Touch Center of Gravity", **axis_cfg),
    yaxis=dict(title="Tackle Center of Gravity", **axis_cfg, scaleanchor="x", scaleratio=1),
    template="plotly_white",
)
fig.add_shape(type="line", x0=-1, x1=1, y0=0, y1=0,
              line=dict(color="lightgray", width=1))
fig.add_shape(type="line", x0=0, x1=0, y0=-1, y1=1,
              line=dict(color="lightgray", width=1))
fig.show()

### 2c. Data-Driven Position Revisions

This data is evidence of where players are *actually* playing - so let's use it to fix many of FBref's misleading positions via clustering, rather than using vague hybrid position labels, which can be often inconsistent and confusing.

In [14]:
# Step 1: Normalize Touch COG per team (0 = most defensive player, 1 = most attacking)
players_filtered["Positional COG"] = (
    players_filtered.groupby("team")["Touch COG"]
    .transform(lambda x: (x - x.min()) / (x.max() - x.min()) if x.max() != x.min() else 0.5)
)

# Step 2: K-Means clustering within each team (3 clusters: Defender / Midfielder / Forward)
def assign_roles(group):
    """Cluster players into Defender/Midfielder/Forward based on within-team Positional COG."""
    X = group[["Positional COG"]].values
    km = KMeans(n_clusters=3, random_state=42, n_init="auto")
    cluster_ids = km.fit_predict(X)
    centers = km.cluster_centers_.flatten()
    order = np.argsort(centers)
    cluster_to_role = {order[0]: "Defender", order[1]: "Midfielder", order[2]: "Forward"}
    return pd.Series([cluster_to_role[c] for c in cluster_ids],
                     index=group.index, name="kmeans_position")

players_filtered["kmeans_position"] = (
    players_filtered.groupby("team", group_keys=False).apply(assign_roles)
)

In [16]:
# Now, let's color our COG map by our new k-means positions
# Flag: position_map is defined in 1b - no redefinition needed here
df = players_filtered.copy()

df['position_numeric'] = df['kmeans_position'].map(position_map)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df["Touch COG"], y=df["Tackle COG"], mode="markers",
    marker=dict(
        size=9, opacity=0.8,
        color=df["position_numeric"],
        colorscale=[[0, '#d62728'], [0.33, '#ff7f0e'],
                    [0.67, '#2ca02c'], [1, '#1f77b4']],
        showscale=True,
        colorbar=dict(title="Position", tickvals=[0,1,2,3],
                      ticktext=['Defender','Wingback','Midfielder','Forward']),
        cmin=0, cmax=3, line=dict(width=0.5, color='white')
    ),
    text=df["player"],
    customdata=np.stack([df["team"], df["kmeans_position"]], axis=-1),
    hovertemplate=(
        "<b>%{text}</b><br>Team: %{customdata[0]}<br>Position: %{customdata[1]}<br>"
        "Touch COG: %{x:.2f}<br>Tackle COG: %{y:.2f}<br><extra></extra>"
    ),
))

teams_unique = sorted(df["team"].unique())
buttons = [dict(label="All", method="update",
                args=[{"x": [df["Touch COG"]], "y": [df["Tackle COG"]]}])]
for t in teams_unique:
    mask = df["team"] == t
    buttons.append(dict(
        label=t, method="update",
        args=[{"x": [df.loc[mask,"Touch COG"]], "y": [df.loc[mask,"Tackle COG"]],
               "text": [df.loc[mask,"player"]],
               "customdata": [np.stack([df.loc[mask,"team"],
                                        df.loc[mask,"kmeans_position"]], axis=-1)]}]
    ))

axis_cfg = dict(range=[-1,1], tickvals=[-1,-0.5,0,0.5,1],
                zeroline=True, zerolinewidth=1, zerolinecolor="gray")
fig.update_layout(
    updatemenus=[dict(buttons=buttons, direction="down", x=0, y=1.15,
                      showactive=True, pad={"r":10})],
    title='Positional COG Map (Colored by K-Means Position)',
    xaxis=dict(title="Touch Center of Gravity", **axis_cfg),
    yaxis=dict(title="Tackle Center of Gravity", **axis_cfg, scaleanchor="x", scaleratio=1),
    template="plotly_white",
)
fig.add_shape(type="line", x0=-1, x1=1, y0=0, y1=0, line=dict(color="lightgray", width=1))
fig.add_shape(type="line", x0=0, x1=0, y0=-1, y1=1, line=dict(color="lightgray", width=1))
fig.show()

In [17]:
# What k-means positions don't yet show: WHO is holding the width
# We can use throw-in frequency as a proxy since wide players take far more throw-ins (this is the best indicator of width
# that FBref provides, since they don't have any actual positional data or heatmaps available at the player level)
width_labels = []

for team, group in df.groupby("team"):
    ti90 = group["Throw-Ins per 90"]
    width = pd.Series("", index=group.index, dtype=object)

    zero_mask = ti90 == 0
    width.loc[zero_mask] = "Central"

    nonzero_mask = ~zero_mask
    group_nz = group[nonzero_mask]

    if len(group_nz) >= 2:
        Xw = group_nz[["Throw-Ins per 90"]].values
        km_w = KMeans(n_clusters=2, random_state=42, n_init="auto")
        cl_ids = km_w.fit_predict(Xw)
        centers_w = km_w.cluster_centers_.flatten()
        order_w = np.argsort(centers_w)
        cluster_to_width = {order_w[0]: "", order_w[1]: "Wide"}
        width.loc[group_nz.index] = [cluster_to_width[c] for c in cl_ids]

    width_labels.append(width.to_frame(name="kmeans_width"))

width_labels = pd.concat(width_labels)
players_filtered["kmeans_width"] = width_labels["kmeans_width"]

In [18]:
def derive_new_position(row):
    # Combines FBref's position labels with our k-means clustering results.
    # Logic:
    # - For clean, unambiguous FBref codes (GK/DF/MF/FW), we'll trust FBref's accuracy of labeling
    # - For multi-position codes (e.g. 'MF,FW'), FBref is too ambiguous,
    #   so we'll defer to our k-means positions.
    # - Flag: Wide Midfielders (k-means = Midfielder AND k-means width = Wide)
    #   will be reclassified as Wingback, regardless of the above.
    pos     = row["Position"]
    knn_pos = row["kmeans_position"]
    width   = row["kmeans_width"]

    # Simple unambiguous FBref codes - we'll give FBref curators the benefit of the doubt here because of the rigitidy
    if pos == "GK":
        new_pos = "Goalkeeper"
    elif pos == "DF":
        new_pos = "Defender"
    elif pos == "MF":
        new_pos = "Midfielder"
    elif pos == "FW":
        new_pos = "Forward"
    # Multi-position string (e.g. 'DF,MF' or 'MF,FW') - Because these position definitions are ambiguous, we'll use k-means
    elif isinstance(pos, str) and len(pos) > 3:
        codes = [p.strip() for p in pos.replace(",", " ").split()]
        if "FW" in codes:
            new_pos = "Forward"
        elif "MF" in codes:
            new_pos = knn_pos
        else:
            new_pos = knn_pos
    else:
        # Fallback for anything unexpected
        new_pos = knn_pos

    # Wide Midfielder = Wingback (applied after all position logic above)
    if knn_pos == "Midfielder" and width == "Wide":
        new_pos = "Wingback"

    return new_pos

players_filtered["new_position"] = players_filtered.apply(derive_new_position, axis=1)
players_filtered["new_position"].value_counts()

new_position
Forward       554
Midfielder    382
Defender      363
Wingback      265
Name: count, dtype: int64

In [21]:
# Final COG map with our ~refined~ positions, including Wingbacks!
# MARKER_STYLE and position_map are imported from 1b constants, so no redefinition needed.
df = players_filtered.copy()
pos_numeric = df["new_position"].map(position_map)  # throwaway series just for visualization

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df["Touch COG"], y=df["Tackle COG"], mode="markers",
    marker=dict(color=pos_numeric, **MARKER_STYLE),
    text=df["player"],
    customdata=np.stack([df["team"], df["new_position"]], axis=-1),
    hovertemplate=(
        "<b>%{text}</b><br>Team: %{customdata[0]}<br>Position: %{customdata[1]}<br>"
        "Touch COG: %{x:.2f}<br>Tackle COG: %{y:.2f}<br><extra></extra>"
    ),
))

teams_unique = sorted(df["team"].unique())
buttons = [dict(
    label="All", method="update",
    args=[{
        "x":          [df["Touch COG"]],
        "y":          [df["Tackle COG"]],
        "text":       [df["player"]],
        "customdata": [np.stack([df["team"], df["new_position"]], axis=-1)],
        "marker":     [dict(color=pos_numeric, **MARKER_STYLE)],
    }]
)]
for t in teams_unique:
    mask = df["team"] == t
    buttons.append(dict(
        label=t, method="update",
        args=[{
            "x":          [df.loc[mask, "Touch COG"]],
            "y":          [df.loc[mask, "Tackle COG"]],
            "text":       [df.loc[mask, "player"]],
            "customdata": [np.stack([df.loc[mask, "team"],
                                     df.loc[mask, "new_position"]], axis=-1)],
            "marker":     [dict(color=pos_numeric[mask], **MARKER_STYLE)],
        }]
    ))

axis_cfg = dict(range=[-1,1], tickvals=[-1,-0.5,0,0.5,1],
                zeroline=True, zerolinewidth=1, zerolinecolor="gray")
fig.update_layout(
    updatemenus=[dict(buttons=buttons, direction="down", x=0, y=1.15,
                      showactive=True, pad={"r":10})],
    title='Positional COG Map (Colored by Derived Position incl. Wingback)',
    xaxis=dict(title="Touch Center of Gravity", **axis_cfg),
    yaxis=dict(title="Tackle Center of Gravity", **axis_cfg, scaleanchor="x", scaleratio=1),
    template="plotly_white",
)
fig.add_shape(type="line", x0=-1, x1=1, y0=0, y1=0, line=dict(color="lightgray", width=1))
fig.add_shape(type="line", x0=0, x1=0, y0=-1, y1=1, line=dict(color="lightgray", width=1))
fig.show()

### 2d. Interactive Player Scatter

In [22]:
# Fully interactive scatter - just pick any two numeric columns for X and Y axes,
# filters by league and position
df = players_filtered.copy()
numeric_cols   = df.select_dtypes(include="number").columns.tolist()
positions_list = sorted(df["new_position"].dropna().unique().tolist())
leagues_list   = sorted(df["league"].dropna().unique().tolist())
leagues_options = [{"label": "All leagues", "value": "ALL"}] + [
    {"label": lg, "value": lg} for lg in leagues_list
]

app = Dash(__name__)
app.layout = html.Div([
    html.H2("Player Scatter (2025/26 League Season)",
            style={"fontFamily": font_family, "fontWeight": "600",
                   "color": "#f5f7fa", "marginBottom": "10px"}),
    html.Div([
        html.Div([
            html.Label("League", style={"fontFamily": font_family, "color": "#f5f7fa"}),
            dcc.Dropdown(id="league_dropdown", options=leagues_options,
                         value="ALL", clearable=False, style={"width": "220px"}),
        ], style={"marginRight": "24px"}),
        html.Div([
            html.Label("Position", style={"fontFamily": font_family, "color": "#f5f7fa"}),
            dcc.Dropdown(id="pos_dropdown",
                         options=[{"label": p, "value": p} for p in positions_list],
                         value=positions_list[0] if positions_list else None,
                         clearable=False, style={"width": "220px"}),
        ], style={"marginRight": "24px"}),
        html.Div([
            html.Label("X-axis metric", style={"fontFamily": font_family, "color": "#f5f7fa"}),
            dcc.Dropdown(id="x_dropdown",
                         options=[{"label": c, "value": c} for c in numeric_cols],
                         value=numeric_cols[0] if numeric_cols else None,
                         searchable=True, style={"width": "260px"}),
        ], style={"marginRight": "24px"}),
        html.Div([
            html.Label("Y-axis metric", style={"fontFamily": font_family, "color": "#f5f7fa"}),
            dcc.Dropdown(id="y_dropdown",
                         options=[{"label": c, "value": c} for c in numeric_cols],
                         value=numeric_cols[1] if len(numeric_cols) > 1 else None,
                         searchable=True, style={"width": "260px"}),
        ]),
    ], style={"display": "flex", "flexDirection": "row",
               "alignItems": "flex-start", "marginBottom": "16px"}),
    dcc.Graph(id="scatter_plot"),
], style={"backgroundColor": "#001f3f", "padding": "20px 24px"})

@app.callback(
    Output("scatter_plot", "figure"),
    Input("league_dropdown", "value"),
    Input("pos_dropdown",    "value"),
    Input("x_dropdown",      "value"),
    Input("y_dropdown",      "value"),
)
def update_scatter(selected_league, selected_pos, x_col, y_col):
    if not (selected_pos and x_col and y_col):
        return px.scatter()
    sub = df.copy()
    if selected_league and selected_league != "ALL":
        sub = sub[sub["league"] == selected_league]
    sub = sub[sub["new_position"] == selected_pos].dropna(subset=[x_col, y_col])
    fig = px.scatter(sub, x=x_col, y=y_col, hover_name="player", color="team")
    fig.update_layout(
        font=dict(family=font_family, color="#f5f7fa"),
        paper_bgcolor="#001f3f", plot_bgcolor="#001f3f",
        legend=dict(font=dict(family=font_family, size=11, color="#f5f7fa"),
                    bgcolor="rgba(0,0,0,0)"),
        xaxis=dict(showgrid=True, gridcolor="#5b6a80"),
        yaxis=dict(showgrid=True, gridcolor="#5b6a80"),
        margin=dict(l=40, r=40, t=40, b=40),
    )
    return fig

app.run(debug=True, use_reloader=False, port=8051)

### 2e. Player Radar Charts

In [23]:
# Feature sets used across radars, KNN, and scoring - defined once here
feature_sets = {
    "Attack": [
        "Non-Penalty Expected Goals per 90",
        "Non-Penalty xG Overperformance",
        "Shots on Target %",
        "Expected Assisted Goals per 90",
        "Goal-Creating Actions per 90",
        "Passes into Penalty Area per 100 Touches",
    ],
    "Progression": [
        "Progressive Passes per 100 Touches",
        "Progressive Passing Distance per 100 Touches",
        "Progressive Carries per 100 Touches",
        "Progressive Carrying Distance per 100 Touches",
        "Take-Ons Attempted per 90",
        "Take-On Success %",
    ],
    "Defense": [
        "Tackles per 90",
        "Dribblers Tackled %",
        "Interceptions per 90",
        "Aerial Win %",
        "Ball Recoveries per 90",
        "Turnovers per 100 Touches",  # inverted in scoring (because less turnovers = better)
    ],
}

all_radar_features = sorted({f for fs in feature_sets.values() for f in fs})

base = players_filtered.dropna(subset=all_radar_features + ["new_position"]).copy()

# Pre-compute within-position percentile ranks for all radar features
df_pct = base.copy()
for feat in all_radar_features:
    df_pct[feat] = df_pct.groupby("new_position")[feat].rank(pct=True)

positions_list = sorted(df_pct["new_position"].unique().tolist())

app = Dash(__name__)
app.layout = html.Div([
    html.H2("Player Radars - 2025/26 Position Percentiles",
            style={"textAlign": "left", "fontFamily": font_family,
                   "fontWeight": "600", "color": "#f5f7fa", "marginBottom": "10px"}),
    html.Div([
        html.Div([
            html.Label("Profile", style={"fontFamily": font_family,
                                         "fontSize": "14px", "fontWeight": "500",
                                         "color": "#f5f7fa", "marginBottom": "4px", "display": "block"}),
            dcc.Dropdown(id="profile_dropdown",
                         options=[{"label": k, "value": k} for k in feature_sets],
                         value="Attack", clearable=False, style={"width": "220px"}),
        ], style={"marginRight": "24px"}),
        html.Div([
            html.Label("Position", style={"fontFamily": font_family,
                                          "fontSize": "14px", "fontWeight": "500",
                                          "color": "#f5f7fa", "marginBottom": "4px", "display": "block"}),
            dcc.Dropdown(id="position_dropdown",
                         options=[{"label": pos, "value": pos} for pos in positions_list],
                         value=positions_list[0] if positions_list else None,
                         clearable=False, style={"width": "220px"}),
        ], style={"marginRight": "24px"}),
        html.Div([
            html.Label("Players (max 3)", style={"fontFamily": font_family,
                                                  "fontSize": "14px", "fontWeight": "500",
                                                  "color": "#f5f7fa", "marginBottom": "4px", "display": "block"}),
            dcc.Dropdown(id="player_dropdown", options=[], multi=True,
                         placeholder="Type player name(s)...", value=[],
                         style={"width": "420px"}),
        ]),
    ], style={"display": "flex", "flexDirection": "row",
               "alignItems": "flex-start", "marginBottom": "16px"}),
    dcc.Graph(id="radar_chart"),
], style={"backgroundColor": "#001f3f", "padding": "20px 24px"})

@app.callback(
    Output("player_dropdown", "options"),
    Output("player_dropdown", "value"),
    Input("position_dropdown", "value"),
)
def update_player_options(selected_position):
    if selected_position is None:
        return [], []
    mask = df_pct["new_position"] == selected_position
    players = sorted(df_pct.loc[mask, "player"].unique().tolist())
    return [{"label": p, "value": p} for p in players], []

@app.callback(
    Output("radar_chart", "figure"),
    Input("profile_dropdown",   "value"),
    Input("position_dropdown",  "value"),
    Input("player_dropdown",    "value"),
)
def update_radar(selected_profile, selected_position, selected_players):
    selected_players = (selected_players or [])[:3]
    categories = feature_sets.get(selected_profile, [])
    fig = go.Figure()
    pos_df = df_pct[df_pct["new_position"] == selected_position] if selected_position else df_pct.iloc[0:0]
    for p in selected_players:
        row = pos_df.loc[pos_df["player"] == p, categories].mean()
        if row.isna().all():
            continue
        fig.add_trace(go.Scatterpolar(r=row.values, theta=categories,
                                       fill="toself", name=p))
    fig.update_layout(
        polar=dict(radialaxis=dict(visible=True, range=[0, 1]),
                   angularaxis=dict(gridcolor="#5b6a80", linecolor="#5b6a80")),
        showlegend=True,
        legend=dict(font=dict(family=font_family, size=12, color="#f5f7fa"),
                    bgcolor="rgba(0,0,0,0)"),
        font=dict(family=font_family, color="#f5f7fa"),
        paper_bgcolor="#001f3f", plot_bgcolor="#001f3f",
        margin=dict(l=40, r=40, t=40, b=40),
    )
    return fig

app.run(debug=True, use_reloader=False, port=8052)

### 2f. Player Similarity Finder (KNN)

In [24]:
# KNN uses the same feature_sets defined in 2e - that cell will have to run before this one
all_knn_features = sorted({f for fs in feature_sets.values() for f in fs})

knn_base = (
    players_filtered
    .dropna(subset=all_knn_features)
    .copy()
    .reset_index(drop=True)
)

scaler     = StandardScaler()
knn_matrix = scaler.fit_transform(knn_base[all_knn_features])

leagues_knn   = sorted(knn_base["league"].unique())
positions_knn = sorted(knn_base["new_position"].dropna().unique())

app = Dash(__name__)
app.layout = html.Div([
    html.H2("Player Similarity Finder - KNN (2025/26)",
            style={"fontFamily": font_family, "fontWeight": "600",
                   "color": "#f5f7fa", "marginBottom": "10px"}),
    html.Div([
        html.Div([
            html.Label("Position", style={"fontFamily": font_family, "color": "#f5f7fa"}),
            dcc.Dropdown(id="knn_position_dropdown",
                         options=[{"label": p, "value": p} for p in positions_knn],
                         value=positions_knn[0] if positions_knn else None,
                         clearable=False, style={"width": "180px"}),
        ], style={"marginRight": "20px"}),
        html.Div([
            html.Label("League", style={"fontFamily": font_family, "color": "#f5f7fa"}),
            dcc.Dropdown(id="knn_league_dropdown",
                         options=[{"label": "All leagues", "value": "ALL"}] +
                                 [{"label": lg, "value": lg} for lg in leagues_knn],
                         value="ALL", clearable=False, style={"width": "220px"}),
        ], style={"marginRight": "20px"}),
        html.Div([
            html.Label("Team", style={"fontFamily": font_family, "color": "#f5f7fa"}),
            dcc.Dropdown(id="knn_team_dropdown", options=[], value="ALL",
                         clearable=False, style={"width": "220px"}),
        ], style={"marginRight": "20px"}),
        html.Div([
            html.Label("Select Player", style={"fontFamily": font_family, "color": "#f5f7fa"}),
            dcc.Dropdown(id="knn_player_dropdown", options=[], value=None,
                         clearable=False, placeholder="Type to search...",
                         style={"width": "280px"}),
        ], style={"marginRight": "20px"}),
        html.Div([
            html.Label("# Similar", style={"fontFamily": font_family, "color": "#f5f7fa"}),
            dcc.Dropdown(id="knn_n_dropdown",
                         options=[{"label": str(n), "value": n} for n in [5, 8, 10, 15]],
                         value=8, clearable=False, style={"width": "100px"}),
        ]),
    ], style={"display": "flex", "flexDirection": "row",
               "alignItems": "flex-start", "flexWrap": "wrap", "marginBottom": "16px"}),
    html.Div(id="knn_player_header",
             style={"fontFamily": font_family, "color": "#f5c518",
                    "fontSize": "16px", "fontWeight": "600", "marginBottom": "8px"}),
    dcc.Graph(id="knn_radar_chart"),
    html.Div(id="knn_results_table", style={"marginTop": "16px"}),
], style={"backgroundColor": "#001f3f", "padding": "20px 24px"})

@app.callback(
    Output("knn_team_dropdown", "options"),
    Output("knn_team_dropdown", "value"),
    Input("knn_league_dropdown",   "value"),
    Input("knn_position_dropdown", "value"),
)
def update_knn_team_options(selected_league, selected_position):
    sub = knn_base.copy()
    if selected_position:
        sub = sub[sub["new_position"] == selected_position]
    if selected_league and selected_league != "ALL":
        sub = sub[sub["league"] == selected_league]
    teams = sorted(sub["team"].unique())
    return ([{"label": "All teams", "value": "ALL"}] +
            [{"label": t, "value": t} for t in teams]), "ALL"

@app.callback(
    Output("knn_player_dropdown", "options"),
    Output("knn_player_dropdown", "value"),
    Input("knn_position_dropdown", "value"),
    Input("knn_league_dropdown",   "value"),
    Input("knn_team_dropdown",     "value"),
)
def update_knn_player_options(selected_position, selected_league, selected_team):
    sub = knn_base.copy()
    if selected_position:
        sub = sub[sub["new_position"] == selected_position]
    if selected_league and selected_league != "ALL":
        sub = sub[sub["league"] == selected_league]
    if selected_team and selected_team != "ALL":
        sub = sub[sub["team"] == selected_team]
    players = sorted(sub["player"].unique())
    return [{"label": p, "value": p} for p in players], None

@app.callback(
    Output("knn_player_header",  "children"),
    Output("knn_radar_chart",    "figure"),
    Output("knn_results_table",  "children"),
    Input("knn_player_dropdown", "value"),
    Input("knn_position_dropdown", "value"),
    Input("knn_n_dropdown",      "value"),
)
def update_knn_results(selected_player, selected_position, n_neighbors):
    empty_fig = go.Figure()
    empty_fig.update_layout(paper_bgcolor="#001f3f", plot_bgcolor="#001f3f",
                             font=dict(color="#f5f7fa", family=font_family))
    if not selected_player or not selected_position:
        return "Select a player above to find similar players.", empty_fig, ""

    pos_df = knn_base[knn_base["new_position"] == selected_position].copy().reset_index(drop=True)
    if selected_player not in pos_df["player"].values:
        return f"'{selected_player}' not found in position '{selected_position}'.", empty_fig, ""

    pos_matrix  = scaler.transform(pos_df[all_knn_features])
    k           = min(n_neighbors + 1, len(pos_df))
    knn_model   = NearestNeighbors(n_neighbors=k, metric="euclidean")
    knn_model.fit(pos_matrix)

    player_loc         = pos_df[pos_df["player"] == selected_player].index[0]
    query_vec          = pos_matrix[player_loc].reshape(1, -1)
    distances, indices = knn_model.kneighbors(query_vec)

    similar_rows = pos_df.iloc[indices[0]].copy()
    similar_rows["similarity_distance"] = distances[0]
    similar_rows = similar_rows[similar_rows["player"] != selected_player].head(n_neighbors)

    radar_df = pos_df.copy()
    for feat in all_knn_features:
        radar_df[feat] = radar_df.groupby("new_position")[feat].rank(pct=True)

    def get_pct(name):
        rows = radar_df[radar_df["player"] == name]
        return None if rows.empty else rows[all_knn_features].mean()

    fig = go.Figure()
    qpct = get_pct(selected_player)
    if qpct is not None:
        fig.add_trace(go.Scatterpolar(r=qpct.values, theta=all_knn_features,
                                       fill="toself", name=selected_player,
                                       line=dict(color="#f5c518", width=2.5), opacity=0.9))
    for i, (_, sim_row) in enumerate(similar_rows.head(3).iterrows()):
        sim_pct = get_pct(sim_row["player"])
        if sim_pct is None:
            continue
        fig.add_trace(go.Scatterpolar(r=sim_pct.values, theta=all_knn_features,
                                       fill="toself", name=sim_row["player"],
                                       line=dict(color=["#1f77b4","#2ca02c","#d62728"][i], width=1.5),
                                       opacity=0.6))
    fig.update_layout(
        polar=dict(radialaxis=dict(visible=True, range=[0,1],
                                    tickfont=dict(color="#f5f7fa", size=9)),
                   angularaxis=dict(gridcolor="#2a3a50", linecolor="#2a3a50",
                                    tickfont=dict(family=font_family, color="#cdd9e5", size=10)),
                   bgcolor="#001f3f"),
        showlegend=True,
        legend=dict(font=dict(family=font_family, size=12, color="#f5f7fa"), bgcolor="rgba(0,0,0,0)"),
        font=dict(family=font_family, color="#f5f7fa"),
        paper_bgcolor="#001f3f", plot_bgcolor="#001f3f",
        margin=dict(l=60, r=60, t=40, b=40),
        title=dict(text=f"Radar: {selected_player} vs. Top 3 Similar {selected_position}s",
                   font=dict(family=font_family, color="#f5f7fa", size=14)),
    )

    query_row   = knn_base[knn_base["player"] == selected_player].iloc[0]
    header_text = (f"Players most similar to {selected_player} "
                   f"({query_row['team']} · {query_row['league']})")

    table_rows = []
    for rank, (_, row) in enumerate(similar_rows.iterrows(), start=1):
        sim_score   = max(0, round((1 - row["similarity_distance"] / 10) * 100, 1))
        age_display = str(row["Age"]).split("-")[0]
        score_color = "#2ca02c" if sim_score >= 80 else "#f5c518" if sim_score >= 60 else "#d62728"
        table_rows.append(html.Tr([
            html.Td(f"#{rank}", style={"color":"#f5c518","fontWeight":"bold","padding":"8px 12px"}),
            html.Td(row["player"],                    style={"color":"#f5f7fa","padding":"8px 12px"}),
            html.Td(row["team"],                      style={"color":"#cdd9e5","padding":"8px 12px"}),
            html.Td(row["league"],                    style={"color":"#cdd9e5","padding":"8px 12px"}),
            html.Td(age_display,                      style={"color":"#cdd9e5","padding":"8px 12px","textAlign":"center"}),
            html.Td(f"{row['Minutes Played']:.0f}",   style={"color":"#cdd9e5","padding":"8px 12px","textAlign":"center"}),
            html.Td(f"{sim_score}%",                  style={"color":score_color,"fontWeight":"bold","padding":"8px 12px","textAlign":"center"}),
        ], style={"borderBottom":"1px solid #1a2a3a"}))

    table = html.Table([
        html.Thead(html.Tr([
            html.Th(col, style={"color":"#a0b4c8","fontFamily":font_family,"padding":"8px 12px",
                                "textAlign":"left","borderBottom":"2px solid #2a3a50","fontWeight":"500"})
            for col in ["Rank","Player","Team","League","Age","Minutes","Similarity"]
        ])),
        html.Tbody(table_rows),
    ], style={"width":"100%","borderCollapse":"collapse","fontFamily":font_family,
              "fontSize":"14px","backgroundColor":"#001f3f"})

    return header_text, fig, table

app.run(debug=True, use_reloader=False, port=8053)

### 2g. Player Scores & Leaderboard

In [25]:
# Weighted category scores (for attack, progression, and defense) out of 100
# Scores are within-position percentile ranks, so a forward's attack score
# is relative to other Forwards only and not all outfield players!
# feature_sets defined in 2e are again reused here.

attack_features = {
    "Non-Penalty Expected Goals per 90":        0.25,
    "Expected Assisted Goals per 90":           0.20,
    "Goal-Creating Actions per 90":             0.20,
    "Non-Penalty xG Overperformance":           0.15,
    "Shots on Target %":                        0.10,
    "Passes into Penalty Area per 100 Touches": 0.10,
}
progression_features = {
    "Progressive Passes per 100 Touches":               0.25,
    "Progressive Carries per 100 Touches":              0.25,
    "Take-On Success %":                                0.15,
    "Take-Ons Attempted per 90":                        0.15,
    "Progressive Passing Distance per 100 Touches":     0.10,
    "Progressive Carrying Distance per 100 Touches":    0.10,
}
defense_features = {
    "Tackles per 90":         0.20,
    "Interceptions per 90":   0.20,
    "Ball Recoveries per 90": 0.20,
    "Dribblers Tackled %":    0.15,
    "Aerial Win %":           0.15,
    "Turnovers per 100 Touches": 0.10,  # inverted, since lower is better
}

invert_stats = {"Turnovers per 100 Touches"}

composite_weights = {
    "Forward":    {"attack": 0.55, "progression": 0.30, "defense": 0.15},
    "Midfielder": {"attack": 0.30, "progression": 0.40, "defense": 0.30},
    "Wingback":   {"attack": 0.25, "progression": 0.40, "defense": 0.35},
    "Defender":   {"attack": 0.15, "progression": 0.25, "defense": 0.60},
}

all_score_features = (list(attack_features) + list(progression_features) + list(defense_features))
missing = [f for f in all_score_features if f not in players_filtered.columns]
if missing:
    print(f"Warning: {len(missing)} features not found:")
    for m in missing:
        print(f"  - {m}")

available_features = [f for f in all_score_features if f in players_filtered.columns]

scores_df = (
    players_filtered
    .dropna(subset=available_features)
    [["player", "team", "league", "season", "new_position", "Minutes Played", "Age"]
     + available_features]
    .copy()
    .reset_index(drop=True)
)

def compute_category_score(df, feature_weights, invert_stats=set()):
    """Within-position percentile rank per feature, then weighted average × 100."""
    weighted_sum = pd.Series(0.0, index=df.index)
    total_weight = pd.Series(0.0, index=df.index)
    for feat, weight in feature_weights.items():
        if feat not in df.columns:
            continue
        ascending = feat not in invert_stats
        percentile = df.groupby("new_position")[feat].rank(pct=True, ascending=ascending)
        weighted_sum += percentile * weight
        total_weight += weight
    return ((weighted_sum / total_weight) * 100).round(1)

scores_df["attack_score"]      = compute_category_score(scores_df, attack_features,      invert_stats)
scores_df["progression_score"] = compute_category_score(scores_df, progression_features, invert_stats)
scores_df["defense_score"]     = compute_category_score(scores_df, defense_features,     invert_stats)

scores_df["Age"] = scores_df["Age"].astype(str).str.split("-").str[0]

print(f"Scores computed for {len(scores_df)} players.")
scores_df[["player","team","new_position","attack_score",
           "progression_score","defense_score"]].sample(3)

Scores computed for 1564 players.


,player,team,new_position,attack_score,progression_score,defense_score
1516,Andrea Pinamonti,Sassuolo,Forward,56.3,8.8,33.9
747,Arthur Avom,Lorient,Midfielder,42.5,75.9,61.6
31,Tyrone Mings,Aston Villa,Defender,43.3,62.3,29.2


In [26]:
# Interactive leaderboard with minimum score sliders for filtering
leagues_lb   = sorted(scores_df["league"].unique())
positions_lb = sorted(scores_df["new_position"].dropna().unique())
score_cols   = {"Attack": "attack_score", "Progression": "progression_score", "Defense": "defense_score"}

app = Dash(__name__)
app.layout = html.Div([
    html.H2("Player Ratings 2025/26",
            style={"fontFamily": font_family, "fontWeight": "600",
                   "color": "#f5f7fa", "marginBottom": "16px"}),
    html.Div([
        html.Div([
            html.Label("League", style={"fontFamily": font_family, "color": "#f5f7fa", "fontSize": "13px"}),
            dcc.Dropdown(id="lb_league",
                         options=[{"label":"All Leagues","value":"ALL"}] +
                                 [{"label":lg,"value":lg} for lg in leagues_lb],
                         value="ALL", clearable=False, style={"width":"220px"}),
        ], style={"marginRight":"20px"}),
        html.Div([
            html.Label("Position", style={"fontFamily": font_family, "color": "#f5f7fa", "fontSize": "13px"}),
            dcc.Dropdown(id="lb_position",
                         options=[{"label":"All Positions","value":"ALL"}] +
                                 [{"label":p,"value":p} for p in positions_lb],
                         value="ALL", clearable=False, style={"width":"180px"}),
        ], style={"marginRight":"20px"}),
        html.Div([
            html.Label("Sort By", style={"fontFamily": font_family, "color": "#f5f7fa", "fontSize": "13px"}),
            dcc.Dropdown(id="lb_sort",
                         options=[{"label":k,"value":v} for k,v in score_cols.items()],
                         value="attack_score", clearable=False, style={"width":"180px"}),
        ], style={"marginRight":"20px"}),
        html.Div([
            html.Label("Direction", style={"fontFamily": font_family, "color": "#f5f7fa", "fontSize": "13px"}),
            dcc.Dropdown(id="lb_direction",
                         options=[{"label":"Top Players","value":"desc"},
                                  {"label":"Bottom Players","value":"asc"}],
                         value="desc", clearable=False, style={"width":"180px"}),
        ], style={"marginRight":"20px"}),
        html.Div([
            html.Label("Show", style={"fontFamily": font_family, "color": "#f5f7fa", "fontSize": "13px"}),
            dcc.Dropdown(id="lb_n",
                         options=[{"label":str(n),"value":n} for n in [10,20,30,50]],
                         value=20, clearable=False, style={"width":"100px"}),
        ]),
    ], style={"display":"flex","flexDirection":"row","alignItems":"flex-start",
               "flexWrap":"wrap","marginBottom":"16px"}),
    html.Div([
        html.Span("Minimum Scores:", style={"fontFamily":font_family,"color":"#a0b4c8",
                                             "fontSize":"13px","marginRight":"20px","lineHeight":"36px"}),
        html.Div([
            html.Label("Attack", style={"fontFamily":font_family,"color":"#f5f7fa","fontSize":"13px"}),
            dcc.Slider(id="lb_min_attack", min=0, max=90, step=5, value=0,
                       marks={i:{"label":str(i),"style":{"color":"#a0b4c8","fontSize":"11px"}}
                              for i in range(0,91,10)},
                       tooltip={"placement":"bottom","always_visible":False}),
        ], style={"width":"220px","marginRight":"32px"}),
        html.Div([
            html.Label("Progression", style={"fontFamily":font_family,"color":"#f5f7fa","fontSize":"13px"}),
            dcc.Slider(id="lb_min_progression", min=0, max=90, step=5, value=0,
                       marks={i:{"label":str(i),"style":{"color":"#a0b4c8","fontSize":"11px"}}
                              for i in range(0,91,10)},
                       tooltip={"placement":"bottom","always_visible":False}),
        ], style={"width":"220px","marginRight":"32px"}),
        html.Div([
            html.Label("Defense", style={"fontFamily":font_family,"color":"#f5f7fa","fontSize":"13px"}),
            dcc.Slider(id="lb_min_defense", min=0, max=90, step=5, value=0,
                       marks={i:{"label":str(i),"style":{"color":"#a0b4c8","fontSize":"11px"}}
                              for i in range(0,91,10)},
                       tooltip={"placement":"bottom","always_visible":False}),
        ], style={"width":"220px"}),
    ], style={"display":"flex","flexDirection":"row","alignItems":"flex-start","flexWrap":"wrap",
               "backgroundColor":"#00162b","padding":"14px 16px","borderRadius":"6px","marginBottom":"16px"}),
    html.Div(id="lb_summary", style={"fontFamily":font_family,"color":"#a0b4c8",
                                      "fontSize":"13px","marginBottom":"12px"}),
    html.Div(id="lb_table"),
], style={"backgroundColor":"#001f3f","padding":"24px"})

def score_color(val):
    if val >= 80: return "#2ca02c"
    elif val >= 65: return "#7fba00"
    elif val >= 50: return "#f5c518"
    elif val >= 35: return "#ff7f0e"
    else: return "#d62728"

@app.callback(
    Output("lb_summary","children"), Output("lb_table","children"),
    Input("lb_league","value"), Input("lb_position","value"),
    Input("lb_sort","value"), Input("lb_direction","value"), Input("lb_n","value"),
    Input("lb_min_attack","value"), Input("lb_min_progression","value"), Input("lb_min_defense","value"),
)
def update_leaderboard(league, position, sort_col, direction, n,
                        min_attack, min_progression, min_defense):
    sub = scores_df.copy()
    if league != "ALL":     sub = sub[sub["league"] == league]
    if position != "ALL":   sub = sub[sub["new_position"] == position]
    sub = sub[(sub["attack_score"] >= min_attack) &
              (sub["progression_score"] >= min_progression) &
              (sub["defense_score"] >= min_defense)]
    sub = sub.sort_values(sort_col, ascending=(direction == "asc")).head(n)

    active = []
    if league != "ALL":        active.append(league)
    if position != "ALL":      active.append(position)
    if min_attack > 0:         active.append(f"Attack >= {min_attack}")
    if min_progression > 0:    active.append(f"Progression >= {min_progression}")
    if min_defense > 0:        active.append(f"Defense >= {min_defense}")
    summary = f"Showing {len(sub)} players" + ("  ·  " + "  ·  ".join(active) if active else "")

    headers = ["Rank","Player","Team","League","Position","Age","Minutes","Attack","Progression","Defense"]
    header_row = html.Tr([html.Th(h, style={
        "fontFamily":font_family,"color":"#a0b4c8","padding":"10px 12px",
        "textAlign":"left" if h in ["Player","Team","League","Position"] else "center",
        "borderBottom":"2px solid #2a3a50","fontWeight":"500","fontSize":"13px","whiteSpace":"nowrap"
    }) for h in headers])

    rows = []
    for rank, (_, row) in enumerate(sub.iterrows(), start=1):
        def cs(col):
            base = {"fontFamily":font_family,"padding":"9px 12px","borderBottom":"1px solid #1a2a3a",
                    "fontSize":"13px","textAlign":"center","fontWeight":"600","color":score_color(row[col])}
            if col == sort_col: base["backgroundColor"] = "rgba(245,197,24,0.08)"
            return base
        ts = {"fontFamily":font_family,"padding":"9px 12px","borderBottom":"1px solid #1a2a3a",
              "fontSize":"13px","color":"#f5f7fa","textAlign":"left"}
        ds = {**ts,"color":"#a0b4c8","textAlign":"center"}
        rows.append(html.Tr([
            html.Td(rank,                              style={**ds,"color":"#f5c518","fontWeight":"600"}),
            html.Td(row["player"],                     style=ts),
            html.Td(row["team"],                       style={**ts,"color":"#cdd9e5"}),
            html.Td(row["league"],                     style=ds),
            html.Td(row["new_position"],               style=ds),
            html.Td(str(row["Age"]),                   style=ds),
            html.Td(f"{row['Minutes Played']:.0f}",    style=ds),
            html.Td(f"{row['attack_score']:.1f}",      style=cs("attack_score")),
            html.Td(f"{row['progression_score']:.1f}", style=cs("progression_score")),
            html.Td(f"{row['defense_score']:.1f}",     style=cs("defense_score")),
        ]))

    table = html.Table([html.Thead(header_row), html.Tbody(rows)],
                       style={"width":"100%","borderCollapse":"collapse","backgroundColor":"#001f3f"})
    return summary, table

app.run(debug=True, use_reloader=False, port=8054)

---

For Part 2 of our exploration into this *sacred* FBref data, let's take the player-level detail and turn it into Team-level analysis.

---

## Chapter 3 - Team Data: Handling & Visualizations

### 3a. Building the Team DataFrame

In [28]:
# Some EDA before we dive into Team analyses: I'd like to understand whether teams'
# distribution of minutes amongst players is normal or skewed.
# Why? Because it will inform whether we calculate our "squad rotation" metric
# using Standard Deviation or Mean Absolute Deviation of minutes!

# Here, we test normality of the "Minutes Played" distribution within each team.
# Shapiro-Wilk is best for small sample sizes (teams typically consist of 20-30 players).
# H0: distribution is normal. p > 0.05 = fail to reject normality.

normality_results = []

for (team, league), group in players_all.groupby(["team", "league"]):
    mins = group["Minutes Played"].dropna()
    if len(mins) < 4:
        continue
    stat, p = stats.shapiro(mins)
    normality_results.append({
        "team":         team,
        "league":       league,
        "n":            len(mins),
        "shapiro_stat": round(stat, 4),
        "p_value":      round(p, 4),
        "normal":       p > 0.05,
    })

normality_df = pd.DataFrame(normality_results)
n_total      = len(normality_df)
n_normal     = normality_df["normal"].sum()
n_non_normal = n_total - n_normal
pct_normal   = round(n_normal / n_total * 100, 1)

print(f"Teams tested:       {n_total}")
print(f"Normal (p > 0.05):  {n_normal}  ({pct_normal}%)")
print(f"Non-normal:         {n_non_normal}  ({round(100 - pct_normal, 1)}%)")
print()
print("By league:")
print(
    normality_df.groupby("league")["normal"].agg(["sum","count"])
    .rename(columns={"sum":"normal","count":"total"})
    .assign(pct_normal=lambda x: (x["normal"]/x["total"]*100).round(1))
)
# -- Verdict --
# Too many teams in our sample are non-normal, thus MAD is preferred over SD.
# rotation_metric = "minutes_mad" is already set in the 1b constants cell.

Teams tested:       96
Normal (p > 0.05):  55  (57.3%)
Non-normal:         41  (42.7%)

By league:
                    normal  total  pct_normal
league                                       
ENG-Premier League       9     20        45.0
ESP-La Liga             13     20        65.0
FRA-Ligue 1              8     18        44.4
GER-Bundesliga          13     18        72.2
ITA-Serie A             12     20        60.0


In [29]:
# Minutes Distribution by Team - interactive histogram colored by derived position.
# MAD band and mean reference line are overlaid to visualize squad rotation.

leagues_list = sorted(players_filtered["league"].dropna().unique())

app = Dash(__name__)
app.layout = html.Div([
    html.H2("Minutes Distribution by Team",
            style={"fontFamily":font_family,"fontWeight":"600",
                   "color":"#f5f7fa","marginBottom":"6px"}),
    html.P(
        "Distribution of total minutes played across all squad members. "
        "A right-skewed distribution indicates a tight core XI; "
        "a flatter distribution indicates broader squad rotation.",
        style={"fontFamily":font_family,"color":"#a0b4c8",
               "fontSize":"13px","marginBottom":"20px"},
    ),
    html.Div([
        html.Div([
            html.Label("League", style={"fontFamily":font_family,
                                        "color":"#f5f7fa","fontSize":"13px"}),
            dcc.Dropdown(
                id="hist_league",
                options=[{"label":lg,"value":lg} for lg in leagues_list],
                value=leagues_list[0],
                clearable=False, style={"width":"240px"},
            ),
        ], style={"marginRight":"24px"}),
        html.Div([
            html.Label("Team", style={"fontFamily":font_family,
                                      "color":"#f5f7fa","fontSize":"13px"}),
            dcc.Dropdown(
                id="hist_team",
                options=[],
                value=None,
                clearable=False, style={"width":"240px"},
            ),
        ]),
    ], style={"display":"flex","flexDirection":"row",
               "alignItems":"flex-start","marginBottom":"20px"}),
    dcc.Graph(id="hist_chart"),
], style={"backgroundColor":"#001f3f","padding":"24px"})


@app.callback(
    Output("hist_team", "options"),
    Output("hist_team", "value"),
    Input("hist_league", "value"),
)
def update_team_options(selected_league):
    teams = sorted(
        players_filtered[players_filtered["league"] == selected_league]["team"].unique()
    )
    return [{"label":t,"value":t} for t in teams], teams[0]


@app.callback(
    Output("hist_chart", "figure"),
    Input("hist_league", "value"),
    Input("hist_team",   "value"),
)
def update_hist(selected_league, selected_team):
    if not selected_team:
        return go.Figure()

    sub = (
        players_filtered[
            (players_filtered["league"] == selected_league) &
            (players_filtered["team"]   == selected_team)
        ]
        [["player", "new_position", "Minutes Played"]]
        .dropna(subset=["Minutes Played"])
        .sort_values("Minutes Played", ascending=False)
        .copy()
    )

    if sub.empty:
        return go.Figure()

    # position_colors defined in 1b constants
    bar_colors = sub["new_position"].map(lambda p: position_colors.get(p, "#a0b4c8"))

    mad  = (sub["Minutes Played"] - sub["Minutes Played"].mean()).abs().mean()
    mean = sub["Minutes Played"].mean()

    fig = go.Figure()
    fig.add_trace(go.Bar(
        x=sub["player"],
        y=sub["Minutes Played"],
        marker=dict(color=bar_colors, line=dict(width=0.5, color="rgba(255,255,255,0.15)")),
        customdata=np.stack([sub["new_position"]], axis=-1),
        hovertemplate=("<b>%{x}</b><br>Position: %{customdata[0]}<br>"
                       "Total Minutes: %{y:.0f}<br><extra></extra>"),
    ))

    fig.add_hline(y=mean, line=dict(color="rgba(245,197,24,0.7)", width=1.5, dash="dash"),
                  annotation_text=f"Mean: {mean:.0f} mins", annotation_position="top right",
                  annotation_font=dict(family=font_family, size=10, color="rgba(245,197,24,0.8)"))
    fig.add_hrect(y0=mean - mad, y1=mean + mad, fillcolor="rgba(245,197,24,0.06)",
                  line=dict(width=0),
                  annotation_text=f"MAD: ±{mad:.0f} mins", annotation_position="top right",
                  annotation_font=dict(family=font_family, size=10, color="rgba(245,197,24,0.5)"))

    legend_items = [("Defender","#d62728"),("Wingback","#ff7f0e"),("Midfielder","#2ca02c"),("Forward","#1f77b4")]
    for j, (pos_label, pos_color) in enumerate(legend_items):
        fig.add_annotation(xref="paper", yref="paper", x=1.01, y=0.98 - j * 0.08,
                           text=f"<span style='color:{pos_color}'>■</span>  {pos_label}",
                           showarrow=False, font=dict(family=font_family, size=11, color="#cdd9e5"),
                           xanchor="left", align="left")

    fig.update_layout(
        title=dict(text=f"{selected_team} - Total Minutes Distribution  |  MAD: {mad:.0f} mins",
                   font=dict(family=font_family, size=15, color="#f5f7fa")),
        xaxis=dict(title=dict(text="Player", font=dict(family=font_family, color="#cdd9e5")),
                   showgrid=False, zeroline=False,
                   tickfont=dict(family=font_family, color="#cdd9e5", size=10), tickangle=-35),
        yaxis=dict(title=dict(text="Total Minutes Played", font=dict(family=font_family, color="#cdd9e5")),
                   showgrid=True, gridcolor="#1a2a3a", zeroline=False,
                   tickfont=dict(family=font_family, color="#cdd9e5")),
        font=dict(family=font_family, color="#f5f7fa"),
        paper_bgcolor="#001f3f", plot_bgcolor="#001f3f",
        showlegend=False, margin=dict(l=60, r=120, t=70, b=100),
    )
    return fig

app.run(debug=True, use_reloader=False, port=8055)

In [30]:
# Aggregate players_all (includes GKs, no minutes filter) up to the team level using groupby.
team_df = (
    players_all
    .groupby(['team', 'league', 'season'], as_index=False)
    .agg(
        nineties_played  = ('90s Played',               'sum'),
        minutes_played   = ('Minutes Played',            'sum'),
        pens_conceded    = ('Penalties Conceded',        'sum'),
        errors           = ('Errors',                   'sum'),
        miscontrols      = ('Miscontrols',               'sum'),
        dispossessed     = ('Dispossessed',              'sum'),
        offsides         = ('Offsides',                  'sum'),
        own_goals        = ('Own Goals',                 'sum'),
        yellow_cards     = ('Yellow Cards',              'sum'),
        red_cards        = ('Red Cards',                 'sum'),
        fouls_committed  = ('Fouls Committed',           'sum'),
        goals            = ('Goals',                    'sum'),
        xg               = ('Expected Goals',            'sum'),
        npxg_overperf    = ('Non-Penalty xG Overperformance', 'sum'),
        minutes_mad      = ('Minutes Played', lambda x: (x - x.mean()).abs().mean()),  # MAD confirmed by Shapiro-Wilk above
))

print(f"team_df: {team_df.shape[0]} team-seasons")
team_df.sample(3)

team_df: 96 team-seasons


,team,league,season,nineties_played,minutes_played,pens_conceded,errors,miscontrols,dispossessed,offsides,own_goals,yellow_cards,red_cards,fouls_committed,goals,xg,npxg_overperf,minutes_mad
12,Bologna,ITA-Serie A,2526,164.8,14820,2,14,215,129,32,0,30,2,222,23,21.2,1.9,263.440329
57,Manchester Utd,ENG-Premier League,2526,175.3,15794,1,11,274,135,25,0,24,1,169,28,31.2,-2.2,358.004535
16,Brighton,ENG-Premier League,2526,175.9,15840,3,9,273,167,29,2,39,0,193,24,25.9,-1.5,472.464000


### 3b. Team Centers of Gravity

In [31]:
# -- Overall team-level Touch COG and Tackle COG --
# Both are minutes-weighted averages across all outfield players.
# Tackle COG uses dropna - players with zero tackles are excluded from the
# tackle average only; they'll still contribute to Touch COG.

overall_touch_cog = (
    players_filtered
    .assign(touch_cog_weighted=lambda x: x["Touch COG"] * x["Minutes Played"])
    .groupby(["team", "league", "season"], as_index=False)
    .agg(sum_touch_weighted=("touch_cog_weighted","sum"),
         sum_minutes_touch=("Minutes Played","sum"))
    .assign(team_touch_cog=lambda x: x["sum_touch_weighted"] / x["sum_minutes_touch"])
    [["team","league","season","team_touch_cog"]]
)

overall_tackle_cog = (
    players_filtered
    .dropna(subset=["Tackle COG"])
    .assign(tackle_cog_weighted=lambda x: x["Tackle COG"] * x["Minutes Played"])
    .groupby(["team","league","season"], as_index=False)
    .agg(sum_tackle_weighted=("tackle_cog_weighted","sum"),
         sum_minutes_tackle=("Minutes Played","sum"))
    .assign(team_tackle_cog=lambda x: x["sum_tackle_weighted"] / x["sum_minutes_tackle"])
    [["team","league","season","team_tackle_cog"]]
)

team_df = team_df.merge(overall_touch_cog,  on=["team","league","season"], how="left")
team_df = team_df.merge(overall_tackle_cog, on=["team","league","season"], how="left")

print(f"Null team_touch_cog:  {team_df['team_touch_cog'].isnull().sum()}")
print(f"Null team_tackle_cog: {team_df['team_tackle_cog'].isnull().sum()}")

Null team_touch_cog:  0
Null team_tackle_cog: 0


In [32]:
# -- Positional Touch COG pivot (Defender / Wingback / Midfielder / Forward) --
team_pos_cog_weighted = (
    players_filtered
    .assign(touch_cog_weighted=players_filtered['Touch COG'] * players_filtered['Minutes Played'])
    .groupby(['team','league','season','new_position'], as_index=False)
    .agg(sum_weighted_cog=('touch_cog_weighted','sum'),
         sum_minutes=('Minutes Played','sum'))
    .assign(avg_touch_cog_weighted=lambda x: x['sum_weighted_cog'] / x['sum_minutes'])
)

team_pos_cog_pivot = (
    team_pos_cog_weighted
    .pivot_table(index=['team','league','season'], columns='new_position',
                 values='avg_touch_cog_weighted')
    .rename(columns=lambda c: f'cog_{c.lower()}')
    .reset_index()
)

team_df = team_df.merge(team_pos_cog_pivot, on=['team','league','season'], how='left')

# Wingback fallback: teams with no classified Wingbacks use Wide Defenders instead
if 'cog_wingback' in team_df.columns and team_df['cog_wingback'].isnull().any():
    wide_defender_cog = (
        players_filtered[
            (players_filtered['new_position'] == 'Defender') &
            (players_filtered['kmeans_width'] == 'Wide')
        ]
        .assign(touch_cog_weighted=lambda x: x['Touch COG'] * x['Minutes Played'])
        .groupby(['team','league','season'], as_index=False)
        .agg(sum_weighted_cog=('touch_cog_weighted','sum'),
             sum_minutes=('Minutes Played','sum'))
        .assign(cog_wingback_fallback=lambda x: x['sum_weighted_cog'] / x['sum_minutes'])
        [['team','league','season','cog_wingback_fallback']]
    )
    team_df = team_df.merge(wide_defender_cog, on=['team','league','season'], how='left')
    team_df['cog_wingback'] = team_df['cog_wingback'].fillna(team_df['cog_wingback_fallback'])
    team_df = team_df.drop(columns=['cog_wingback_fallback'])

# Fallback for any remaining null positional COG
cog_cols = [c for c in team_df.columns if c.startswith('cog_')]
pos_map  = {'cog_defender': 'Defender', 'cog_midfielder': 'Midfielder',
            'cog_forward': 'Forward', 'cog_wingback': 'Wingback'}

for col in cog_cols:
    if not team_df[col].isnull().any():
        continue
    position = pos_map.get(col)
    if position is None:
        continue
    pos_fallback = (
        players_filtered[players_filtered['new_position'] == position]
        .assign(touch_cog_weighted=lambda x: x['Touch COG'] * x['Minutes Played'])
        .groupby(['team','league','season'], as_index=False)
        .agg(sum_weighted_cog=('touch_cog_weighted','sum'),
             sum_minutes=('Minutes Played','sum'))
        .assign(**{f'{col}_fallback': lambda x: x['sum_weighted_cog'] / x['sum_minutes']})
        [['team','league','season', f'{col}_fallback']]
    )
    team_df = team_df.merge(pos_fallback, on=['team','league','season'], how='left')
    team_df[col] = team_df[col].fillna(team_df[f'{col}_fallback'])
    team_df = team_df.drop(columns=[f'{col}_fallback'])

null_counts = team_df[cog_cols].isnull().sum()
if null_counts.any():
    print("Remaining nulls in COG columns:")
    print(null_counts[null_counts > 0])
else:
    print("No nulls in COG columns!")

# -- Sanity check: Let's see the Premier League teams sorted by Defender COG now --
team_df[['team','league','season'] + cog_cols].query(
    "league == 'ENG-Premier League'"
).sort_values('cog_defender', ascending=False).head(20)

No nulls in COG columns!


,team,league,season,cog_defender,cog_forward,cog_midfielder,cog_wingback
51,Liverpool,ENG-Premier League,2526,-0.106693,0.350345,0.000041,0.022549
13,Bournemouth,ENG-Premier League,2526,-0.209588,0.222779,0.023376,-0.077043
57,Manchester Utd,ENG-Premier League,2526,-0.23252,0.229334,0.140352,0.062121
56,Manchester City,ENG-Premier League,2526,-0.246143,0.321171,0.060792,-0.030428
2,Arsenal,ENG-Premier League,2526,-0.250572,0.331094,0.042629,0.056572
95,Wolves,ENG-Premier League,2526,-0.264107,0.168497,-0.095515,0.010513
64,Newcastle Utd,ENG-Premier League,2526,-0.285426,0.302496,0.036463,-0.019445
20,Chelsea,ENG-Premier League,2526,-0.296942,0.326832,0.036829,-0.03481
16,Brighton,ENG-Premier League,2526,-0.310282,0.234074,-0.054046,-0.025832
46,Leeds United,ENG-Premier League,2526,-0.312085,0.255249,0.022551,-0.049737


### 3c. Multi-Team COG Comparisons

In [33]:
# -- COG Spine Chart: compare up to 8 teams' positional COG profiles --
# position_order and cog_col_map are imported from 1b constants.

spine_df      = team_df.dropna(subset=list(cog_col_map.values())).copy()
leagues_spine = sorted(spine_df["league"].unique())

app = Dash(__name__)
app.layout = html.Div([
    html.H2("Team Position Group COG",
            style={"fontFamily":font_family,"fontWeight":"600",
                   "color":"#f5f7fa","marginBottom":"10px"}),
    html.Div([
        html.Div([
            html.Label("League", style={"fontFamily":font_family,"color":"#f5f7fa","fontSize":"13px"}),
            dcc.Dropdown(id="spine_league",
                         options=[{"label":lg,"value":lg} for lg in leagues_spine],
                         value=leagues_spine[0], clearable=False, style={"width":"240px"}),
        ], style={"marginRight":"24px"}),
        html.Div([
            html.Label("Teams (max 8)", style={"fontFamily":font_family,"color":"#f5f7fa","fontSize":"13px"}),
            dcc.Dropdown(id="spine_teams", options=[], multi=True,
                         placeholder="Select teams to compare...", style={"width":"500px"}),
        ]),
    ], style={"display":"flex","flexDirection":"row","alignItems":"flex-start","marginBottom":"16px"}),
    dcc.Graph(id="spine_chart"),
], style={"backgroundColor":"#001f3f","padding":"24px"})

@app.callback(Output("spine_teams","options"), Output("spine_teams","value"),
              Input("spine_league","value"))
def update_spine_teams(league):
    teams = sorted(spine_df[spine_df["league"]==league]["team"].unique())
    return [{"label":t,"value":t} for t in teams], teams[:5]

@app.callback(Output("spine_chart","figure"),
              Input("spine_league","value"), Input("spine_teams","value"))
def update_spine(selected_league, selected_teams):
    fig = go.Figure()
    if not selected_teams:
        fig.update_layout(paper_bgcolor="#001f3f", plot_bgcolor="#001f3f"); return fig
    selected_teams = selected_teams[:8]
    sub    = spine_df[(spine_df["league"]==selected_league) & (spine_df["team"].isin(selected_teams))]
    colors = px.colors.qualitative.Plotly + px.colors.qualitative.D3
    tcolors = {t: colors[i%len(colors)] for i,t in enumerate(selected_teams)}
    league_sub  = spine_df[spine_df["league"]==selected_league]
    league_avgs = {pos: league_sub[col].mean() for pos,col in cog_col_map.items()}

    fig.add_trace(go.Scatter(
        x=list(range(len(position_order))),
        y=[league_avgs[p] for p in position_order],
        mode="lines+markers", name="League Avg",
        line=dict(color="rgba(160,180,200,0.4)",width=2,dash="dot"),
        marker=dict(size=8,color="rgba(160,180,200,0.4)"),
        hovertemplate="<b>League Average</b><br>%{text}: %{y:.3f}<br><extra></extra>",
        text=position_order,
    ))
    for team in selected_teams:
        row = sub[sub["team"]==team]
        if row.empty: continue
        row = row.iloc[0]; color = tcolors[team]
        fig.add_trace(go.Scatter(
            x=list(range(len(position_order))),
            y=[row[cog_col_map[p]] for p in position_order],
            mode="lines+markers", name=team,
            line=dict(color=color,width=2.5),
            marker=dict(size=12,color=color,line=dict(width=1.5,color="white")),
            hovertemplate=f"<b>{team}</b><br>%{{text}}: %{{y:.3f}}<br><extra></extra>",
            text=position_order,
        ))

    fig.add_hline(y=0, line=dict(color="rgba(255,255,255,0.15)",width=1,dash="dot"),
                  annotation_text="Midfield", annotation_position="right",
                  annotation_font=dict(family=font_family,size=10,color="rgba(200,215,230,0.5)"))
    fig.update_layout(
        xaxis=dict(tickvals=list(range(len(position_order))), ticktext=position_order,
                   showgrid=False, zeroline=False,
                   tickfont=dict(family=font_family,color="#cdd9e5",size=13)),
        yaxis=dict(title=dict(text="Touch COG  ←  Defensive  |  Attacking  →",
                              font=dict(family=font_family,color="#cdd9e5")),
                   range=[-1,1], showgrid=True, gridcolor="#1a2a3a", zeroline=False,
                   tickfont=dict(family=font_family,color="#cdd9e5")),
        title=dict(text=f"Positional COG Spines - {selected_league}",
                   font=dict(family=font_family,size=16,color="#f5f7fa")),
        font=dict(family=font_family,color="#f5f7fa"),
        paper_bgcolor="#001f3f", plot_bgcolor="#001f3f",
        legend=dict(font=dict(family=font_family,size=11,color="#f5f7fa"),bgcolor="rgba(0,0,0,0)"),
        hovermode="x unified", margin=dict(l=60,r=60,t=60,b=40),
    )
    return fig

app.run(debug=True, use_reloader=False, port=8056)

In [34]:
# -- Side-by-Side Pitch Comparison (4-3-3 template, any two teams) --
# cog_col_map is imported from 1b constants.

pitch_df = team_df.dropna(subset=list(cog_col_map.values())).copy()
all_teams_sorted = (pitch_df[["team","league"]].drop_duplicates()
                    .sort_values(["league","team"]))
team_options = [{"label":f"{r['team']}  ({r['league']})", "value":r["team"]}
                for _,r in all_teams_sorted.iterrows()]

PITCH_LENGTH = 120; PITCH_WIDTH = 80; CENTER_X = PITCH_WIDTH/2; SPACING = 12

def cog_to_y(cog): return (cog+1)/2*PITCH_LENGTH
def pitch_y_to_cog(y): return (y/PITCH_LENGTH)*2-1

def get_positions(row):
    pos = []
    def_y = cog_to_y(row["cog_defender"])
    for x in [CENTER_X-SPACING/2, CENTER_X+SPACING/2]:
        pos.append({"name":"Defender","group":"Defender","x":x,"y":def_y,
                    "cog":row["cog_defender"],"color":"#d62728"})
    wb_y = cog_to_y(row["cog_wingback"])
    pos += [{"name":"Wingback (L)","group":"Wingback","x":10,"y":wb_y,
              "cog":row["cog_wingback"],"color":"#ff7f0e"},
             {"name":"Wingback (R)","group":"Wingback","x":PITCH_WIDTH-10,"y":wb_y,
              "cog":row["cog_wingback"],"color":"#ff7f0e"}]
    mid_y = cog_to_y(row["cog_midfielder"])
    for x in [CENTER_X-SPACING, CENTER_X, CENTER_X+SPACING]:
        pos.append({"name":"Midfielder","group":"Midfielder","x":x,"y":mid_y,
                    "cog":row["cog_midfielder"],"color":"#2ca02c"})
    fwd_y = cog_to_y(row["cog_forward"])
    for x in [CENTER_X-SPACING, CENTER_X, CENTER_X+SPACING]:
        pos.append({"name":"Forward","group":"Forward","x":x,"y":fwd_y,
                    "cog":row["cog_forward"],"color":"#1f77b4"})
    return pos

def build_pitch(row, x_offset, show_legend=True):
    shapes=[]; traces=[]; W=PITCH_WIDTH; L=PITCH_LENGTH; cx=x_offset+CENTER_X
    def s(t,**kw): return dict(type=t,layer="below",**kw)
    shapes += [
        s("rect",x0=x_offset,y0=0,x1=x_offset+W,y1=L,fillcolor="#2d5d2d",line=dict(width=0)),
        s("rect",x0=x_offset,y0=0,x1=x_offset+W,y1=L,line=dict(color="white",width=2),fillcolor="rgba(0,0,0,0)"),
        s("line",x0=x_offset,y0=L/2,x1=x_offset+W,y1=L/2,line=dict(color="white",width=2)),
        s("circle",x0=cx-10,y0=L/2-10,x1=cx+10,y1=L/2+10,line=dict(color="white",width=2),fillcolor="rgba(0,0,0,0)"),
        s("circle",x0=cx-0.8,y0=L/2-0.8,x1=cx+0.8,y1=L/2+0.8,fillcolor="white",line=dict(width=0)),
        s("rect",x0=x_offset+18,y0=0,x1=x_offset+W-18,y1=18,line=dict(color="white",width=2),fillcolor="rgba(0,0,0,0)"),
        s("rect",x0=x_offset+18,y0=L-18,x1=x_offset+W-18,y1=L,line=dict(color="white",width=2),fillcolor="rgba(0,0,0,0)"),
        s("rect",x0=x_offset+30,y0=0,x1=x_offset+W-30,y1=6,line=dict(color="white",width=2),fillcolor="rgba(0,0,0,0)"),
        s("rect",x0=x_offset+30,y0=L-6,x1=x_offset+W-30,y1=L,line=dict(color="white",width=2),fillcolor="rgba(0,0,0,0)"),
    ]
    for y_spot in [12, L-12]:
        shapes.append(s("circle",x0=cx-0.8,y0=y_spot-0.8,x1=cx+0.8,y1=y_spot+0.8,
                         fillcolor="white",line=dict(width=0)))
    seen=set()
    for p in get_positions(row):
        show = show_legend and p["group"] not in seen; seen.add(p["group"])
        traces.append(go.Scatter(
            x=[p["x"]+x_offset], y=[p["y"]], mode="markers",
            marker=dict(size=36,color=p["color"],line=dict(color="white",width=3),opacity=0.92),
            name=p["group"], legendgroup=p["group"], showlegend=show,
            hovertemplate=f"<b>{p['name']}</b><br>COG: {p['cog']:.3f}<br>Pitch Y: {p['y']:.1f}<br><extra></extra>",
        ))
    return shapes, traces

app = Dash(__name__)
d1 = all_teams_sorted.iloc[0]["team"]; d2 = all_teams_sorted.iloc[1]["team"]
app.layout = html.Div([
    html.H2("Team COG Comparison - 4-3-3 Template",
            style={"fontFamily":font_family,"fontWeight":"600","color":"#f5f7fa","marginBottom":"6px"}),
    html.P("Compare two teams' positional Centers of Gravity side by side. "
           "Teams from different leagues can be selected.",
           style={"fontFamily":font_family,"color":"#a0b4c8","fontSize":"13px","marginBottom":"16px"}),
    html.Div([
        html.Div([
            html.Label("Left Team", style={"fontFamily":font_family,"color":"#f5f7fa","fontSize":"13px"}),
            dcc.Dropdown(id="team_left", options=team_options, value=d1,
                         clearable=False, searchable=True, style={"width":"340px"}),
        ], style={"marginRight":"40px"}),
        html.Div([
            html.Label("Right Team", style={"fontFamily":font_family,"color":"#f5f7fa","fontSize":"13px"}),
            dcc.Dropdown(id="team_right", options=team_options, value=d2,
                         clearable=False, searchable=True, style={"width":"340px"}),
        ]),
    ], style={"display":"flex","flexDirection":"row","alignItems":"flex-start","marginBottom":"16px"}),
    dcc.Graph(id="pitch_compare_chart", style={"height":"760px"}),
], style={"backgroundColor":"#001f3f","padding":"24px"})

@app.callback(Output("pitch_compare_chart","figure"),
              Input("team_left","value"), Input("team_right","value"))
def update_comparison(team_left, team_right):
    GAP = 18; fig = go.Figure()
    row_l = pitch_df[pitch_df["team"]==team_left].iloc[0]
    row_r = pitch_df[pitch_df["team"]==team_right].iloc[0]
    x_off_r = PITCH_WIDTH + GAP
    shapes_l, traces_l = build_pitch(row_l, x_offset=0, show_legend=True)
    shapes_r, traces_r = build_pitch(row_r, x_offset=x_off_r, show_legend=False)
    for sh in shapes_l + shapes_r:
        fig.add_shape(**{k:v for k,v in sh.items() if k!="type"}, type=sh["type"])
    for tr in traces_l + traces_r: fig.add_trace(tr)

    annotations = []
    for cog_val in [-0.8,-0.6,-0.4,-0.2,0.0,0.2,0.4,0.6,0.8]:
        y_val = cog_to_y(cog_val); label = f"{cog_val:+.1f}"
        fig.add_shape(type="line",x0=-2,x1=0,y0=y_val,y1=y_val,
                      line=dict(color="rgba(220,235,250,0.5)",width=1.5))
        annotations.append(dict(x=-3.5,y=y_val,text=label,showarrow=False,
                                  font=dict(family=font_family,size=9,color="rgba(220,235,250,0.65)"),
                                  xanchor="right",yanchor="middle"))
        rx = x_off_r + PITCH_WIDTH
        fig.add_shape(type="line",x0=rx,x1=rx+2,y0=y_val,y1=y_val,
                      line=dict(color="rgba(220,235,250,0.5)",width=1.5))
        annotations.append(dict(x=rx+3.5,y=y_val,text=label,showarrow=False,
                                  font=dict(family=font_family,size=9,color="rgba(220,235,250,0.65)"),
                                  xanchor="left",yanchor="middle"))
    annotations += [
        dict(x=PITCH_WIDTH/2, y=PITCH_LENGTH+4,
             text=f"<b>{team_left}</b><br><span style='font-size:11px'>{row_l['league']}</span>",
             showarrow=False, font=dict(family=font_family,size=14,color="#f5f7fa"),
             xanchor="center", yanchor="bottom"),
        dict(x=x_off_r+PITCH_WIDTH/2, y=PITCH_LENGTH+4,
             text=f"<b>{team_right}</b><br><span style='font-size:11px'>{row_r['league']}</span>",
             showarrow=False, font=dict(family=font_family,size=14,color="#f5f7fa"),
             xanchor="center", yanchor="bottom"),
    ]
    fig.update_layout(
        annotations=annotations,
        xaxis=dict(range=[-10,x_off_r+PITCH_WIDTH+10],showgrid=False,zeroline=False,showticklabels=False),
        yaxis=dict(range=[-4,PITCH_LENGTH+12],showgrid=False,zeroline=False,showticklabels=False,
                   scaleanchor="x",scaleratio=1),
        plot_bgcolor="#1a1a1a", paper_bgcolor="#001f3f",
        font=dict(family=font_family,color="#f5f7fa"),
        legend=dict(x=0.5,y=-0.04,xanchor="center",yanchor="top",orientation="h",
                    bgcolor="rgba(0,0,0,0)",font=dict(family=font_family,size=12,color="#f5f7fa")),
        hovermode="closest", margin=dict(l=55,r=55,t=30,b=20), height=760,
    )
    return fig

app.run(debug=True, use_reloader=False, port=8057)

### 3d. Touch COG vs. Tackle COG Scatter

In [36]:
# Team-level Touch COG vs. Tackle COG - one dot per team per league.
#
# Both axes are league-relative *z-scores*, so the scatter is centered around 0
# for each league. This removes the systematic negative bias in Tackle COG
# (tackles naturally happen more in defensive zones across all teams) and
# makes the question "does this team engage higher or deeper *relative to peers*"
# much more readable in this visualization.

scatter_df = team_df.dropna(subset=["team_touch_cog","team_tackle_cog","xg"]).copy()
leagues_sc = sorted(scatter_df["league"].unique())

app = Dash(__name__)
app.layout = html.Div([
    html.H2("Team Touch COG vs. Tackle COG (League-Relative Z-Scores)",
            style={"fontFamily":font_family,"fontWeight":"600",
                   "color":"#f5f7fa","marginBottom":"6px"}),
    html.P(
        "Each dot is a team, colored by total xG. Axes show league-relative z-scores - 0 = league average, "
        "+1 = one std above average. "
        "X axis = Touch COG (where players live on the pitch). "
        "Y axis = Tackle COG (where they win the ball).",
        style={"fontFamily":font_family,"color":"#a0b4c8",
               "fontSize":"13px","marginBottom":"16px"}),
    html.Div([
        html.Label("League", style={"fontFamily":font_family,
                                    "color":"#f5f7fa","fontSize":"13px"}),
        dcc.Dropdown(
            id="tc_league",
            options=[{"label":lg,"value":lg} for lg in leagues_sc],
            value=leagues_sc[0], clearable=False, style={"width":"280px"}),
    ], style={"marginBottom":"16px"}),
    dcc.Graph(id="tc_scatter"),
], style={"backgroundColor":"#001f3f","padding":"24px"})

@app.callback(Output("tc_scatter","figure"), Input("tc_league","value"))
def update_tc_scatter(selected_league):
    sub = scatter_df[scatter_df["league"] == selected_league].copy()
    sub["touch_cog_z"]  = (sub["team_touch_cog"] - sub["team_touch_cog"].mean()) / sub["team_touch_cog"].std()
    sub["tackle_cog_z"] = (sub["team_tackle_cog"] - sub["team_tackle_cog"].mean()) / sub["team_tackle_cog"].std()

    fig = go.Figure()
    fig.add_hline(y=0, line=dict(color="#2a3a50", width=1, dash="dot"))
    fig.add_vline(x=0, line=dict(color="#2a3a50", width=1, dash="dot"))

    for qx, qy, label in [
        ( 1.2,  1.2, "High Touch<br>High Tackle"),
        (-1.2,  1.2, "Low Touch<br>High Tackle"),
        ( 1.2, -1.2, "High Touch<br>Low Tackle"),
        (-1.2, -1.2, "Low Touch<br>Low Tackle"),
    ]:
        fig.add_annotation(x=qx, y=qy, text=label, showarrow=False,
                           font=dict(family=font_family, size=9,
                                     color="rgba(160,180,200,0.35)"),
                           align="center")

    fig.add_trace(go.Scatter(
        x=sub["touch_cog_z"], y=sub["tackle_cog_z"],
        mode="markers+text", text=sub["team"],
        textposition="top center",
        textfont=dict(family=font_family, size=10, color="#cdd9e5"),
        marker=dict(size=12, color=sub["xg"], colorscale="RdYlGn",
                    cmin=sub["xg"].min(), cmax=sub["xg"].max(), showscale=True,
                    colorbar=dict(title=dict(text="Total xG",
                                             font=dict(family=font_family, color="#cdd9e5", size=11)),
                                  tickfont=dict(family=font_family, color="#cdd9e5", size=10),
                                  thickness=12, len=0.6),
                    line=dict(width=1.5, color="white"), opacity=0.92),
        hovertemplate=("<b>%{text}</b><br>Touch COG z: %{x:.2f}<br>"
                       "Tackle COG z: %{y:.2f}<br>xG: %{marker.color:.1f}<br><extra></extra>"),
    ))

    fig.update_layout(
        xaxis=dict(title=dict(text="Touch COG (z-score)  ←  Below Average  |  Above Average  →",
                               font=dict(family=font_family, color="#cdd9e5")),
                   showgrid=True, gridcolor="#1a2a3a", zeroline=False,
                   tickfont=dict(family=font_family, color="#cdd9e5")),
        yaxis=dict(title=dict(text="Tackle COG (z-score)  ←  Below Average  |  Above Average  →",
                               font=dict(family=font_family, color="#cdd9e5")),
                   showgrid=True, gridcolor="#1a2a3a", zeroline=False,
                   tickfont=dict(family=font_family, color="#cdd9e5")),
        title=dict(text=f"Team Touch vs. Tackle COG (League Z-Scores) - {selected_league}",
                   font=dict(family=font_family, size=16, color="#f5f7fa")),
        font=dict(family=font_family, color="#f5f7fa"),
        paper_bgcolor="#001f3f", plot_bgcolor="#001f3f",
        showlegend=False, hovermode="closest",
        margin=dict(l=60, r=100, t=60, b=60),
    )
    return fig

app.run(debug=True, use_reloader=False, port=8058)

### 3e. Effect of Tactics & Rotation on Output

In [37]:
# -- Multi-variable Team Scatter -- Attacking Compactness, MAD, Touch/Tackle COG vs xG/Goals/Mistakes --

# With this interactive visuailization, we can explore all these Team-level different relationships in one place!

# Pre-compute attacking compactness and add to team_df if not already there
if "attacking_compactness" not in team_df.columns:
    team_df["attacking_compactness"] = team_df["cog_forward"] - team_df["cog_defender"]

# Combine mistake components into one column
team_df["mistakes"] = (
    team_df["errors"] +
    team_df["miscontrols"] +
    team_df["dispossessed"] +
    team_df["pens_conceded"]
)

x_options = {
    "Attacking Compactness":  "attacking_compactness",
    "Squad Rotation (MAD)":   "minutes_mad",
    "Touch COG":              "team_touch_cog",
    "Tackle COG":             "team_tackle_cog",
}
y_options = {
    "xGoals":    "xg",
    "Goals":     "goals",
    "Mistakes":  "mistakes",
}

leagues_list = sorted(team_df["league"].dropna().unique())
colors       = px.colors.qualitative.Plotly

app = Dash(__name__)
app.layout = html.Div([
    html.H2(id="b1_title",
            style={"fontFamily":font_family,"fontWeight":"600",
                   "color":"#f5f7fa","marginBottom":"6px"}),
    html.Div([
        html.Div([
            html.Label("X Axis", style={"fontFamily":font_family,"color":"#f5f7fa","fontSize":"13px"}),
            dcc.Dropdown(id="b1_x",
                         options=[{"label":k,"value":k} for k in x_options],
                         value="Attacking Compactness",
                         clearable=False, style={"width":"240px"}),
        ], style={"marginRight":"24px"}),
        html.Div([
            html.Label("Y Axis", style={"fontFamily":font_family,"color":"#f5f7fa","fontSize":"13px"}),
            dcc.Dropdown(id="b1_y",
                         options=[{"label":k,"value":k} for k in y_options],
                         value="xGoals",
                         clearable=False, style={"width":"180px"}),
        ], style={"marginRight":"24px"}),
        html.Div([
            html.Label("League", style={"fontFamily":font_family,"color":"#f5f7fa","fontSize":"13px"}),
            dcc.Dropdown(id="b1_league",
                         options=[{"label":"All Leagues","value":"ALL"}] +
                                 [{"label":lg,"value":lg} for lg in leagues_list],
                         value="ALL",
                         clearable=False, style={"width":"220px"}),
        ]),
    ], style={"display":"flex","flexDirection":"row",
               "alignItems":"flex-start","marginBottom":"20px"}),
    dcc.Graph(id="b1_scatter"),
], style={"backgroundColor":"#001f3f","padding":"24px"})

@app.callback(
    Output("b1_title",   "children"),
    Output("b1_scatter", "figure"),
    Input("b1_x",        "value"),
    Input("b1_y",        "value"),
    Input("b1_league",   "value"),
)
def update_b1(x_label, y_label, selected_league):
    x_col = x_options[x_label]
    y_col = y_options[y_label]
    title = f"Effect of {x_label} on {y_label}"
    sub = team_df.dropna(subset=[x_col, y_col]).copy()
    if selected_league != "ALL":
        sub = sub[sub["league"] == selected_league]
    fig = go.Figure()
    for i, league in enumerate(sub["league"].unique()):
        lg_sub = sub[sub["league"] == league]
        color  = colors[i % len(colors)]
        fig.add_trace(go.Scatter(
            x=lg_sub[x_col], y=lg_sub[y_col],
            mode="markers+text", name=league,
            text=lg_sub["team"], textposition="top center",
            textfont=dict(family=font_family, size=9, color="rgba(200,215,230,0.7)"),
            marker=dict(size=10, color=color, line=dict(width=0.5, color="white")),
            hovertemplate=("<b>%{text}</b><br>" + f"{x_label}: %{{x:.3f}}<br>" +
                           f"{y_label}: %{{y:.2f}}<br><extra></extra>"),
        ))
    if len(sub) >= 2:
        z  = np.polyfit(sub[x_col].astype(float), sub[y_col].astype(float), 1)
        p  = np.poly1d(z)
        xr = np.linspace(sub[x_col].min(), sub[x_col].max(), 100)
        fig.add_trace(go.Scatter(x=xr, y=p(xr), mode="lines", name="Trend", showlegend=True,
                                 line=dict(color="rgba(245,197,24,0.7)", width=2, dash="dot"),
                                 hoverinfo="skip"))
    fig.update_layout(
        xaxis=dict(title=dict(text=x_label, font=dict(family=font_family, color="#cdd9e5")),
                   showgrid=True, gridcolor="#2a3a50", zeroline=False,
                   tickfont=dict(family=font_family, color="#cdd9e5")),
        yaxis=dict(title=dict(text=y_label, font=dict(family=font_family, color="#cdd9e5")),
                   showgrid=True, gridcolor="#2a3a50", zeroline=False,
                   tickfont=dict(family=font_family, color="#cdd9e5")),
        font=dict(family=font_family, color="#f5f7fa"),
        paper_bgcolor="#001f3f", plot_bgcolor="#001f3f",
        legend=dict(font=dict(family=font_family, size=11, color="#f5f7fa"), bgcolor="rgba(0,0,0,0)"),
        margin=dict(l=60, r=40, t=40, b=60),
    )
    return title, fig

app.run(debug=True, use_reloader=False, port=8059)

### 3f. Correlation Heatmap - What Correlates Best with xG?

In [38]:
# Do any of these team statistics correlate with Expected Goals (xG)? Hopefully this cell can visualize that answer

# -- Variable universe --
corr_vars = {
    "Forward COG":            "cog_forward",
    "Midfielder COG":         "cog_midfielder",
    "Defender COG":           "cog_defender",
    "Wingback COG":           "cog_wingback",
    "Team Touch COG":         "team_touch_cog",
    "Team Tackle COG":        "team_tackle_cog",
    "Attacking Compactness":  "attacking_compactness",
    "Squad Rotation":         "minutes_mad",
    "Yellow Cards":           "yellow_cards",
    "Fouls Committed":        "fouls_committed",
}

TARGET = "xg"

corr_df = team_df.dropna(subset=list(corr_vars.values()) + [TARGET]).copy()

results = []
for label, col in corr_vars.items():
    r, p = spearmanr(corr_df[col].astype(float), corr_df[TARGET].astype(float))
    results.append({"label": label, "col": col, "r": round(r, 3), "p": round(p, 4),
                    "abs_r": abs(round(r, 3)), "sig": p < 0.05})

results_df = pd.DataFrame(results).sort_values("abs_r", ascending=True)

print("Spearman correlations with xG:")
print(results_df.sort_values("abs_r", ascending=False)[["label", "r", "p", "sig"]].to_string(index=False))

leagues_list  = sorted(corr_df["league"].dropna().unique())
colors_league = {lg: px.colors.qualitative.Plotly[i % len(px.colors.qualitative.Plotly)]
                 for i, lg in enumerate(leagues_list)}

app = Dash(__name__)
app.layout = html.Div([
    html.H2("What Correlates Best with xG?",
            style={"fontFamily":font_family,"fontWeight":"600",
                   "color":"#f5f7fa","marginBottom":"6px"}),
    html.P(
        "Spearman rank correlation - robust to outliers and non-normal distributions. "
        "Bars show correlation strength with xG. Select any variable to explore its scatter.",
        style={"fontFamily":font_family,"color":"#a0b4c8",
               "fontSize":"13px","marginBottom":"20px"},
    ),
    html.Div([
        html.Label("Explore Variable vs. xG",
                   style={"fontFamily":font_family,"color":"#f5f7fa","fontSize":"13px"}),
        dcc.Dropdown(id="b2_var",
                     options=[{"label":k,"value":k} for k in corr_vars],
                     value=results_df.iloc[-1]["label"],
                     clearable=False, style={"width":"280px"}),
    ], style={"marginBottom":"20px"}),
    html.Div([
        html.Div([dcc.Graph(id="b2_bar",     style={"height":"520px"})],
                 style={"width":"45%","marginRight":"2%"}),
        html.Div([dcc.Graph(id="b2_scatter", style={"height":"520px"})],
                 style={"width":"53%"}),
    ], style={"display":"flex","flexDirection":"row","alignItems":"flex-start"}),
], style={"backgroundColor":"#001f3f","padding":"24px"})

@app.callback(
    Output("b2_bar",     "figure"),
    Output("b2_scatter", "figure"),
    Input("b2_var",      "value"),
)
def update_b2(selected_label):
    selected_col = corr_vars[selected_label]
    bar_colors = []
    for _, row in results_df.iterrows():
        if row["label"] == selected_label:      bar_colors.append("#f5c518")
        elif not row["sig"]:                    bar_colors.append("rgba(120,140,160,0.4)")
        elif row["r"] >= 0:                     bar_colors.append("#2ca02c")
        else:                                    bar_colors.append("#d62728")

    bar_fig = go.Figure(go.Bar(
        x=results_df["r"], y=results_df["label"], orientation="h",
        marker=dict(color=bar_colors, line=dict(width=0.5, color="rgba(255,255,255,0.2)")),
        hovertemplate="<b>%{y}</b><br>Spearman r: %{x:.3f}<br><extra></extra>",
    ))
    bar_fig.add_vline(x=0, line=dict(color="rgba(255,255,255,0.2)", width=1))
    for _, row in results_df.iterrows():
        bar_fig.add_annotation(
            x=row["r"] + (0.015 if row["r"] >= 0 else -0.015), y=row["label"],
            text="*" if row["sig"] else "", showarrow=False,
            font=dict(family=font_family, size=13, color="#f5c518"),
            xanchor="left" if row["r"] >= 0 else "right",
        )
    bar_fig.update_layout(
        title=dict(text="Spearman r with xG  (* = p < 0.05)",
                   font=dict(family=font_family, size=14, color="#f5f7fa")),
        xaxis=dict(title=dict(text="Spearman r", font=dict(family=font_family, color="#cdd9e5")),
                   range=[-1, 1], showgrid=True, gridcolor="#1a2a3a", zeroline=False,
                   tickfont=dict(family=font_family, color="#cdd9e5")),
        yaxis=dict(showgrid=False, zeroline=False,
                   tickfont=dict(family=font_family, color="#cdd9e5", size=11)),
        font=dict(family=font_family, color="#f5f7fa"),
        paper_bgcolor="#001f3f", plot_bgcolor="#001f3f",
        margin=dict(l=160, r=40, t=60, b=40), showlegend=False,
    )

    scatter_sub = corr_df.dropna(subset=[selected_col, TARGET]).copy()
    r_val = results_df[results_df["label"] == selected_label]["r"].values[0]
    p_val = results_df[results_df["label"] == selected_label]["p"].values[0]
    scatter_fig = go.Figure()
    for league in leagues_list:
        lg_sub = scatter_sub[scatter_sub["league"] == league]
        if lg_sub.empty: continue
        scatter_fig.add_trace(go.Scatter(
            x=lg_sub[selected_col], y=lg_sub[TARGET],
            mode="markers+text", name=league,
            text=lg_sub["team"], textposition="top center",
            textfont=dict(family=font_family, size=8, color="rgba(200,215,230,0.6)"),
            marker=dict(size=9, color=colors_league[league], line=dict(width=0.5, color="white")),
            hovertemplate=("<b>%{text}</b><br>" + f"{selected_label}: %{{x:.3f}}<br>" +
                           "xG: %{y:.1f}<br><extra></extra>"),
        ))
    if len(scatter_sub) >= 2:
        z  = np.polyfit(scatter_sub[selected_col].astype(float),
                        scatter_sub[TARGET].astype(float), 1)
        p  = np.poly1d(z)
        xr = np.linspace(scatter_sub[selected_col].min(), scatter_sub[selected_col].max(), 100)
        scatter_fig.add_trace(go.Scatter(x=xr, y=p(xr), mode="lines", name="Trend",
                                         showlegend=False,
                                         line=dict(color="rgba(245,197,24,0.6)", width=2, dash="dot"),
                                         hoverinfo="skip"))
    sig_text = "p < 0.05" if p_val < 0.05 else "p ≥ 0.05 (not significant)"
    scatter_fig.update_layout(
        title=dict(text=f"{selected_label} vs. xG  |  r = {r_val:.3f}  |  {sig_text}",
                   font=dict(family=font_family, size=13, color="#f5f7fa")),
        xaxis=dict(title=dict(text=selected_label, font=dict(family=font_family, color="#cdd9e5")),
                   showgrid=True, gridcolor="#2a3a50", zeroline=False,
                   tickfont=dict(family=font_family, color="#cdd9e5")),
        yaxis=dict(title=dict(text="Total xG", font=dict(family=font_family, color="#cdd9e5")),
                   showgrid=True, gridcolor="#2a3a50", zeroline=False,
                   tickfont=dict(family=font_family, color="#cdd9e5")),
        font=dict(family=font_family, color="#f5f7fa"),
        paper_bgcolor="#001f3f", plot_bgcolor="#001f3f",
        legend=dict(font=dict(family=font_family, size=10, color="#f5f7fa"), bgcolor="rgba(0,0,0,0)"),
        margin=dict(l=60, r=40, t=60, b=60),
    )
    return bar_fig, scatter_fig

app.run(debug=True, use_reloader=False, port=8060)

Spearman correlations with xG:
                label      r      p   sig
       Team Touch COG  0.624 0.0000  True
      Team Tackle COG  0.572 0.0000  True
       Midfielder COG  0.524 0.0000  True
          Forward COG  0.473 0.0000  True
         Defender COG  0.425 0.0000  True
         Wingback COG  0.389 0.0001  True
         Yellow Cards -0.122 0.2359 False
      Fouls Committed -0.099 0.3391 False
Attacking Compactness  0.046 0.6591 False
       Squad Rotation  0.032 0.7565 False


### 3g. Statistical Significance - Mann-Whitney U Tests

In [39]:
# Mann-Whitney U Tests: does Touch COG or Squad Rotation significantly affect xG?
# rotation_metric and rotation_label are defined in the 1b constants cell.

mw_df = team_df.dropna(subset=["team_touch_cog", rotation_metric, "xg"]).copy()

touch_median = mw_df["team_touch_cog"].median()
mad_median   = mw_df[rotation_metric].median()

mw_df["touch_group"] = np.where(
    mw_df["team_touch_cog"] >= touch_median, "High Touch COG", "Low Touch COG"
)
mw_df["mad_group"] = np.where(
    mw_df[rotation_metric] >= mad_median, "High MAD (Stable XI)", "Low MAD (High Rotation)"
)

def run_mw(df, group_col, value_col="xg"):
    groups  = df[group_col].unique()
    g1_vals = df[df[group_col] == groups[0]][value_col]
    g2_vals = df[df[group_col] == groups[1]][value_col]
    stat, p = mannwhitneyu(g1_vals, g2_vals, alternative="two-sided")
    return {
        "group_a":     groups[0], "group_b":     groups[1],
        "median_a":    round(g1_vals.median(), 2), "median_b":    round(g2_vals.median(), 2),
        "n_a":         len(g1_vals), "n_b":         len(g2_vals),
        "U_stat":      round(stat, 2), "p_value":     round(p, 4),
        "significant": p < 0.05,
        "g1_vals":     g1_vals, "g2_vals":     g2_vals,
    }

touch_mw = run_mw(mw_df, "touch_group")
mad_mw   = run_mw(mw_df, "mad_group")

for label, res in [("Touch COG vs xG", touch_mw), ("MAD vs xG", mad_mw)]:
    sig = "Significant" if res["significant"] else "Not significant"
    print(f"-- {label} --")
    print(f"  {res['group_a']} (n={res['n_a']}):  median xG = {res['median_a']}")
    print(f"  {res['group_b']} (n={res['n_b']}):  median xG = {res['median_b']}")
    print(f"  U = {res['U_stat']},  p = {res['p_value']}  →  {sig}")
    print()

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=["Touch COG vs xG",
                    f"Squad Rotation ({rotation_label}) vs xG"],
    horizontal_spacing=0.12,
)

def add_violin_pair(fig, result, col):
    palette = {"high": "#2ca02c", "low": "#d62728"}
    for i, (group, vals, color_key) in enumerate([
        (result["group_a"], result["g1_vals"], "high"),
        (result["group_b"], result["g2_vals"], "low"),
    ]):
        color = palette[color_key]
        fig.add_trace(go.Violin(
            y=vals, name=group, side="both", meanline_visible=True,
            fillcolor=f"rgba({int(color[1:3],16)},{int(color[3:5],16)},{int(color[5:7],16)},0.25)",
            line=dict(color=color, width=1.5),
            points="all", pointpos=0,
            marker=dict(size=5, color=color, opacity=0.6, line=dict(width=0.5, color="white")),
            box_visible=True,
            box=dict(fillcolor=f"rgba({int(color[1:3],16)},{int(color[3:5],16)},{int(color[5:7],16)},0.4)",
                     line=dict(color=color, width=1)),
            hovertemplate=f"<b>{group}</b><br>xG: %{{y:.1f}}<br><extra></extra>",
            showlegend=False, x0=group,
        ), row=1, col=col)

    y_max  = max(result["g1_vals"].max(), result["g2_vals"].max())
    y_ann  = y_max * 1.08
    p_text = (f"p = {result['p_value']} (significant)" if result["significant"]
              else f"p = {result['p_value']} (not significant)")
    p_color = "#2ca02c" if result["significant"] else "#d62728"
    fig.add_annotation(
        xref=f"x{col if col > 1 else ''} domain", yref=f"y{col if col > 1 else ''}",
        x=0.5, y=y_ann, text=p_text, showarrow=False,
        font=dict(family=font_family, size=12, color=p_color), align="center",
    )
    fig.add_shape(
        type="line", xref=f"x{col if col > 1 else ''} domain",
        yref=f"y{col if col > 1 else ''}",
        x0=0.2, x1=0.8, y0=y_ann * 0.995, y1=y_ann * 0.995,
        line=dict(color="rgba(200,215,230,0.4)", width=1),
    )

add_violin_pair(fig, touch_mw, col=1)
add_violin_pair(fig, mad_mw,   col=2)

for col, res in [(1, touch_mw), (2, mad_mw)]:
    diff = round(res["median_a"] - res["median_b"], 2)
    diff_text  = f"Median diff: {diff:+.2f} xG"
    diff_color = "#2ca02c" if diff > 0 else "#d62728"
    fig.add_annotation(
        xref=f"x{col if col > 1 else ''} domain", yref="paper",
        x=0.5, y=-0.10, text=diff_text, showarrow=False,
        font=dict(family=font_family, size=11, color=diff_color), align="center",
    )

fig.update_layout(
    title=dict(
        text=("Mann-Whitney U Tests - Does Touch COG or Squad Rotation "
              "significantly affect xG?<br>"
              "<sup>Groups split at median. Two-sided test. p < 0.05 = significant.</sup>"),
        font=dict(family=font_family, size=15, color="#f5f7fa"), x=0, xanchor="left",
    ),
    font=dict(family=font_family, color="#f5f7fa"),
    paper_bgcolor="#001f3f", plot_bgcolor="#001f3f",
    showlegend=False, height=580,
    margin=dict(l=60, r=40, t=120, b=80),
    violingap=0.3, violinmode="group",
)
for ax in ["xaxis", "xaxis2"]:
    fig.layout[ax].update(showgrid=False, zeroline=False,
                          tickfont=dict(family=font_family, color="#cdd9e5", size=11))
for ax in ["yaxis", "yaxis2"]:
    fig.layout[ax].update(
        title=dict(text="Total xG", font=dict(family=font_family, color="#cdd9e5")),
        showgrid=True, gridcolor="#1a2a3a", zeroline=False,
        tickfont=dict(family=font_family, color="#cdd9e5"),
    )
fig.show()

-- Touch COG vs xG --
  Low Touch COG (n=48):  median xG = 17.7
  High Touch COG (n=48):  median xG = 22.9
  U = 461.0,  p = 0.0  →  Significant

-- MAD vs xG --
  Low MAD (High Rotation) (n=48):  median xG = 20.0
  High MAD (Stable XI) (n=48):  median xG = 21.45
  U = 1047.0,  p = 0.4438  →  Not significant

